In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!git clone https://github.com/RIA-lab/MMTD.git

In [ ]:
%cd /kaggle/working/MMTD

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
from transformers import BertModel, ViTModel
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss

class MMTD(nn.Module):
    def __init__(self, num_labels=2):
        super(MMTD, self).__init__()
        
        # Enkoderlar: BERT ve DiT (Document Image Transformer)
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("microsoft/dit-base")
        
        # 🔥 ASİMETRİK YAPI: Metni Dondur
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        
        self.hidden_size = 768
        
        # 🌟 YENİ: CONFIDENCE ESTIMATORS (Güven Tahminleyiciler)
        # Metin ve görselin kendi içindeki gürültü/kalite oranını 0-1 arası puanlar
        self.text_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Linear(self.hidden_size // 2, 1),
            nn.Sigmoid()
        )
        
        self.image_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Linear(self.hidden_size // 2, 1),
            nn.Sigmoid()
        )
        
        # 🔥 BIDIRECTIONAL CROSS-ATTENTION
        self.t2i_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        self.i2t_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        # 🔥 DİNAMİK GATING (Her iki yön için)
        self.gate_t2i = nn.Sequential(nn.Linear(self.hidden_size * 2, self.hidden_size), nn.Sigmoid())
        self.gate_i2t = nn.Sequential(nn.Linear(self.hidden_size * 2, self.hidden_size), nn.Sigmoid())
        
        self.layer_norm = nn.LayerNorm(self.hidden_size)
        self.pooler = nn.Sequential(nn.Linear(self.hidden_size, self.hidden_size), nn.Tanh())
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, modality_dropout_prob=0.1, **kwargs):
        input_ids = input_ids.clamp(0, 30521)
        
        text_last = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        image_last = self.image_encoder(pixel_values=pixel_values).last_hidden_state

        # 🔥 MODALITY DROPOUT (Gürültü Direnci)
        if self.training:
            rand_val = torch.rand(1).item()
            if rand_val < modality_dropout_prob:
                image_last = torch.zeros_like(image_last)
            elif rand_val < 2 * modality_dropout_prob:
                text_last = torch.zeros_like(text_last)

        # 🌟 YENİ: CONFIDENCE SKORLARININ HESAPLANMASI
        # Her bir kelime ve resim yaması (patch) için 0 ile 1 arasında güven skoru üretilir
        t_conf = self.text_confidence(text_last)   # Boyut: [Batch, Seq_Len, 1]
        i_conf = self.image_confidence(image_last) # Boyut: [Batch, Seq_Len, 1]

        # Güvensiz verilerin sesini kısıyoruz (Filtreleme)
        text_confident = text_last * t_conf
        image_confident = image_last * i_conf

        # 🔥 BIDIRECTIONAL FUSION (Artık Güven Filtreli)
        # Text-to-Image Attention (Metin, sadece güvendiği resim parçalarına bakar)
        attn_t2i, _ = self.t2i_attention(query=text_last, key=image_confident, value=image_confident)
        gate_t = self.gate_t2i(torch.cat([text_last, attn_t2i], dim=-1))
        fused_text = text_last + (gate_t * attn_t2i)
        
        # Image-to-Text Attention (Resim, sadece güvendiği metin parçalarına bakar)
        attn_i2t, _ = self.i2t_attention(query=image_last, key=text_confident, value=text_confident)
        gate_i = self.gate_i2t(torch.cat([image_last, attn_i2t], dim=-1))
        fused_image = image_last + (gate_i * attn_i2t)

        # Final Combine (Metin yolu üzerinden pooling)
        combined = self.layer_norm(fused_text + fused_image[:, :fused_text.size(1), :])
        
        pooled = self.pooler(combined[:, 0, :])
        logits = self.classifier(self.dropout(pooled))

        loss = None
        if labels is not None:
            labels = labels.clamp(0, 1).long()
            loss = CrossEntropyLoss()(logits.view(-1, 2), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
# Email_dataset.py dosyasındaki sürüm uyumsuzluğunu çözen yama
file_path = '/kaggle/working/MMTD/Email_dataset.py'

with open(file_path, 'r') as file:
    file_data = file.read()

# Eski ismi yeni isimle değiştir
file_data = file_data.replace('BeitFeatureExtractor', 'BeitImageProcessor')

# Düzeltilmiş kodu dosyaya geri yaz
with open(file_path, 'w') as file:
    file.write(file_data)

print("✅ Yama uygulandı! BeitFeatureExtractor başarıyla BeitImageProcessor olarak değiştirildi.")

In [ ]:
# Email_dataset.py dosyasındaki CLIP sürüm uyumsuzluğunu çözen yama
file_path = '/kaggle/working/MMTD/Email_dataset.py'

with open(file_path, 'r') as file:
    file_data = file.read()

# Eski ismi yeni isimle değiştir
file_data = file_data.replace('CLIPFeatureExtractor', 'CLIPImageProcessor')

# Düzeltilmiş kodu dosyaya geri yaz
with open(file_path, 'w') as file:
    file.write(file_data)

print("✅ Yama 2 uygulandı! CLIPFeatureExtractor başarıyla CLIPImageProcessor olarak değiştirildi.")

In [ ]:
import os
import glob

# Projedeki tüm Python dosyalarını bul
proje_dizini = '/kaggle/working/MMTD/**/*.py'
python_dosyalari = glob.glob(proje_dizini, recursive=True)

degisen_dosya_sayisi = 0

for dosya_yolu in python_dosyalari:
    with open(dosya_yolu, 'r', encoding='utf-8') as dosya:
        icerik = dosya.read()
    
    # Eğer dosyada 'FeatureExtractor' kelimesi geçiyorsa değiştir
    if 'FeatureExtractor' in icerik:
        yeni_icerik = icerik.replace('FeatureExtractor', 'ImageProcessor')
        
        with open(dosya_yolu, 'w', encoding='utf-8') as dosya:
            dosya.write(yeni_icerik)
            
        print(f"✅ Temizlendi: {dosya_yolu.split('/')[-1]}")
        degisen_dosya_sayisi += 1

print(f"\n🚀 OPERASYON TAMAM! Toplam {degisen_dosya_sayisi} dosyada tüm isimler tek seferde güncellendi.")

In [ ]:
# Eksik olan torchtext ve onun yardımcı kütüphanesi portalocker'ı kuruyoruz
!pip install torchtext portalocker


In [ ]:
# utils.py içindeki sistemi çökerten gereksiz GloVe importunu silen yama
file_path = '/kaggle/working/MMTD/utils.py'

with open(file_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()

with open(file_path, 'w', encoding='utf-8') as file:
    for line in lines:
        # Eğer satırda torchtext geçmiyorsa dosyaya geri yaz (Yani torchtext'i çöpe at)
        if 'torchtext' not in line:
            file.write(line)

print("✅ Ameliyat başarılı! Gereksiz torchtext bağımlılığı koddan tamamen söküldü.")

In [ ]:
import os
import glob

# Projedeki tüm Python dosyalarını bul
proje_dizini = '/kaggle/working/MMTD/**/*.py'
python_dosyalari = glob.glob(proje_dizini, recursive=True)

temizlenen_dosya_sayisi = 0

for dosya_yolu in python_dosyalari:
    with open(dosya_yolu, 'r', encoding='utf-8') as file:
        lines = file.readlines()
        
    yazilacak_satirlar = []
    degisiklik_var_mi = False
    
    for line in lines:
        if 'torchtext' not in line:
            yazilacak_satirlar.append(line)
        else:
            degisiklik_var_mi = True
            
    # Sadece değişiklik yaptıysak dosyaya geri yaz (diski yormayalım)
    if degisiklik_var_mi:
        with open(dosya_yolu, 'w', encoding='utf-8') as file:
            file.writelines(yazilacak_satirlar)
        print(f"🧹 Temizlendi: {dosya_yolu.split('/')[-1]}")
        temizlenen_dosya_sayisi += 1

print(f"\n✅ OPERASYON TAMAM! Toplam {temizlenen_dosya_sayisi} dosyadan torchtext kalıntıları tamamen silindi.")

In [ ]:
# models.py içindeki sınıf adını main.py'nin beklediği isme (MMTD) çeviren yama
file_path = '/kaggle/working/MMTD/models.py'

with open(file_path, 'r', encoding='utf-8') as f:
    icerik = f.read()

# Sınıf adlarını değiştir
icerik = icerik.replace('class PhishGuard_MMTD', 'class MMTD')
icerik = icerik.replace('super(PhishGuard_MMTD', 'super(MMTD')

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(icerik)

print("✅ İsim etiketi düzeltildi! Yeni Asimetrik motorumuz artık sistemin tanıdığı 'MMTD' adını taşıyor.")

In [ ]:
# 1. main.py Klasör Hatası Bypass Yaması
main_path = '/kaggle/working/MMTD/main.py'
with open(main_path, 'r', encoding='utf-8') as f:
    main_code = f.read()

# Olmayan klasörleri atla, sahte bir döngü yarat
main_code = main_code.replace('os.listdir(bert_checkpoint_path)', '["fold_1"]')
main_code = main_code.replace('os.listdir(beit_checkpoint_path)', '["fold_1"]')

with open(main_path, 'w', encoding='utf-8') as f:
    f.write(main_code)

# 2. models.py HuggingFace Doğrudan İndirme Yaması
models_path = '/kaggle/working/MMTD/models.py'
with open(models_path, 'r', encoding='utf-8') as f:
    models_code = f.read()

# Değişkenleri yoksay, doğrudan orijinal modellerin ismini yazdır
models_code = models_code.replace('BertForSequenceClassification.from_pretrained(bert_pretrain_weight)', 'BertForSequenceClassification.from_pretrained("bert-base-uncased")')
models_code = models_code.replace('BeitForImageClassification.from_pretrained(beit_pretrain_weight)', 'BeitForImageClassification.from_pretrained("microsoft/beit-base-patch16-224-pt22k")')

with open(models_path, 'w', encoding='utf-8') as f:
    f.write(models_code)

print("✅ SİSTEM HACKLENDİ: Model artık yerel klasör aramayacak, doğrudan HuggingFace'ten taptaze BERT ve BEiT indirecek!")

In [ ]:
# main.py'yi kandırmak için sahte (içi boş) klasörleri ve dosyaları oluşturuyoruz
!mkdir -p ./output/bert/checkpoints/fold_1
!mkdir -p ./output/beit/checkpoints/fold_1

# İçlerine sahte model ve config dosyaları koyalım (0 byte boyutunda)
!touch ./output/bert/checkpoints/fold_1/pytorch_model.bin
!touch ./output/bert/checkpoints/fold_1/config.json

!touch ./output/beit/checkpoints/fold_1/pytorch_model.bin
!touch ./output/beit/checkpoints/fold_1/config.json

print("✅ KUKLA DOSYALAR YERLEŞTİRİLDİ! main.py artık bu sahte dosyaları bulup mutlu olacak.")

In [ ]:
import os

# main.py'nin arayabileceği tüm olası eski modellerin listesi
eski_modeller = ['bert', 'beit', 'dit', 'clip', 'vilt']

for model in eski_modeller:
    klasor_yolu = f'./output/{model}/checkpoints/fold_1'
    os.makedirs(klasor_yolu, exist_ok=True)
    
    # İçlerine sahte model ve config dosyaları koyalım
    open(f'{klasor_yolu}/pytorch_model.bin', 'w').close()
    open(f'{klasor_yolu}/config.json', 'w').close()

print("✅ GLOBAL KUKLA OPERASYONU TAMAM! Tüm eski hayalet klasörler sahte dosyalarla dolduruldu.")

In [ ]:
# main.py içindeki sürüm hatasını (evaluation_strategy -> eval_strategy) düzelten yama
file_path = '/kaggle/working/MMTD/main.py'

with open(file_path, 'r', encoding='utf-8') as file:
    file_data = file.read()

# Eski argüman ismini yenisiyle değiştir
file_data = file_data.replace('evaluation_strategy', 'eval_strategy')

with open(file_path, 'w', encoding='utf-8') as file:
    file.write(file_data)

print("✅ Yama uygulandı! 'evaluation_strategy' başarıyla 'eval_strategy' olarak güncellendi.")

In [ ]:
# main.py içindeki TrainingArguments hatalarını gideren toplu yama
file_path = '/kaggle/working/MMTD/main.py'

with open(file_path, 'r', encoding='utf-8') as file:
    file_data = file.read()

# 1. Strateji ismini güncelle
file_data = file_data.replace('evaluation_strategy', 'eval_strategy')

# 2. Hata veren 'overwrite_output_dir' argümanını devre dışı bırak 
# (Zaten biz elle klasör temizliği yaptığımız için buna ihtiyacımız yok)
file_data = file_data.replace('overwrite_output_dir = True', 'overwrite_output_dir = False')
file_data = file_data.replace('overwrite_output_dir=True', 'overwrite_output_dir=False')

with open(file_path, 'w', encoding='utf-8') as file:
    file.write(file_data)

print("✅ Yama uygulandı! main.py artık modern TrainingArguments ile uyumlu.")

In [ ]:
import os

file_path = '/kaggle/working/MMTD/main.py'

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Hata çıkaran eski TrainingArguments bloğunu tamamen hedef alıyoruz.
# eval_strategy ve diğer güncel parametrelerle tertemiz bir blok yazıyoruz.

old_block_start = "args = TrainingArguments("
# Bu kısım genellikle ')' ile biter. O bloğu bulup değiştireceğiz.

new_args_block = """args = TrainingArguments(
    output_dir=args.output_dir,
    eval_strategy="epoch",  # Modern isim
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=10,
    remove_unused_columns=False, # Multimodal modellerde bu kritiktir!
    push_to_hub=False,
    report_to="none" # WandB sormasın diye garantili çözüm
)"""

# Basit bir replace yerine, daha güvenli bir yöntemle o kısmı yamalayalım
import re
# args = TrainingArguments(...) bloğunu yakalayan regex (parantez içini de alır)
content = re.sub(r"args = TrainingArguments\(.*?\)", new_args_block, content, flags=re.DOTALL)

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ KRİTİK YAMA TAMAMLANDI: TrainingArguments modern sürüme göre baştan yazıldı.")

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import torch
import pandas as pd
from transformers import TrainingArguments, Trainer
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD # Bizim asimetrik modelimiz
from utils import SplitData

# 1. PARAMETRELER
output_dir = './output/PhishGuard_Asymmetric'
os.makedirs(output_dir, exist_ok=True)
os.environ["WANDB_MODE"] = "disabled"

# 2. VERİ SETİNİ HAZIRLA
# Fold 1 üzerinden hızlıca eğitimi başlatıyoruz
split_data = SplitData('DATA/email_data/EDP.csv', 1)
train_df, val_df = split_data.train_df, split_data.test_df

train_dataset = EDPDataset(train_df, 'DATA/email_data/images')
val_dataset = EDPDataset(val_df, 'DATA/email_data/images')
collator = EDPCollator()

# 3. ASİMETRİK MODELİ YÜKLE
# Modellerimiz doğrudan HuggingFace'ten gelecek şekilde ayarlandı
model = MMTD()

# 4. EĞİTİM ARGÜMANLARI (Modern Versiyon)
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir=os.path.join(output_dir, 'logs'),
    logging_steps=10,
    remove_unused_columns=False,
    report_to="none"
)

# 5. TRAINER BAŞLAT
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

print("🚀 EĞİTİM BAŞLIYOR...")
trainer.train()
trainer.save_model(output_dir)
print(f"✅ EĞİTİM TAMAMLANDI! Model {output_dir} klasörüne kaydedildi.")

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import torch
import pandas as pd
from transformers import TrainingArguments, Trainer
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD 
from utils import SplitData

# 1. PARAMETRELER VE ORTAM
output_dir = './output/PhishGuard_Asymmetric'
os.makedirs(output_dir, exist_ok=True)
os.environ["WANDB_MODE"] = "disabled"

# 2. VERİ SETİNİ HAZIRLA
# Not: Orijinal repo 'train' ve 'test' isimlerini kullanıyor.
split_data = SplitData('DATA/email_data/EDP.csv', 1)
train_df, val_df = split_data.train, split_data.test  # DÜZELTME BURADA

train_dataset = EDPDataset(train_df, 'DATA/email_data/images')
val_dataset = EDPDataset(val_df, 'DATA/email_data/images')
collator = EDPCollator()

# 3. ASİMETRİK MODELİ YÜKLE
# Bu kısım zaten HuggingFace'ten çekecek şekilde modellerini hazırlamıştık.
model = MMTD()

# 4. EĞİTİM ARGÜMANLARI (Modern Kaggle/Transformers Uyumu)
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir=os.path.join(output_dir, 'logs'),
    logging_steps=10,
    remove_unused_columns=False, # Multimodal için kritik: Sütunları silme
    report_to="none"
)

# 5. TRAINER (EĞİTMEN) KURULUMU
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

print("🚀 TÜM ENGELLER AŞILDI, EĞİTİM BAŞLIYOR...")
trainer.train()
trainer.save_model(output_dir)
print(f"✅ ZAFER! Model {output_dir} klasörüne kaydedildi.")

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import torch
import pandas as pd
from transformers import TrainingArguments, Trainer
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD 
from utils import SplitData, metrics

# 1. ORTAM AYARLARI
output_dir = './output/PhishGuard_Asymmetric'
os.makedirs(output_dir, exist_ok=True)
os.environ["WANDB_MODE"] = "disabled"

# 2. VERİ SETİNİ HAZIRLA (Orijinal Callable Mantığı)
# Fold 1 için veriyi bölüyoruz
split_data = SplitData('DATA/email_data/EDP.csv', 5) # Orijinaldeki gibi 5 fold hazırlığı
train_df, test_df = split_data() # İŞTE KRİTİK DÜZELTME: Nesneyi fonksiyon gibi çağırıyoruz

# Orijinal kodunda path önce, df sonra geliyor: EDPDataset('path', df)
train_dataset = EDPDataset('DATA/email_data/pics', train_df)
test_dataset = EDPDataset('DATA/email_data/pics', test_df)
collator = EDPCollator()

# 3. ASİMETRİK MODELİ YÜKLE
# models.py dosyasını HuggingFace'ten çekecek şekilde hacklemiştik, 
# bu yüzden parametre göndermemize gerek yok.
model = MMTD()

# 4. EĞİTİM ARGÜMANLARI (Modern Kaggle Uyumu)
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",      # evaluation_strategy yerine modern isim
    save_strategy="epoch",
    learning_rate=5e-4,         # Orijinal kodundaki learning rate
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.0,
    load_best_model_at_end=True,
    logging_steps=10,
    remove_unused_columns=False,
    report_to="none"
)

# 5. TRAINER KURULUMU
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=collator,
    compute_metrics=metrics,    # Orijinal metrics fonksiyonunu bağladık
)

print("🚀 SİSTEM ANALİZ EDİLDİ VE ATEŞLENİYOR...")
trainer.train()
trainer.save_model(output_dir)
print(f"✅ EĞİTİM TAMAM! Model {output_dir} klasöründe.")

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import torch
from transformers import TrainingArguments, Trainer
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD 
from utils import SplitData, metrics

os.environ["WANDB_MODE"] = "disabled"

# MUTLAK YOLLAR
csv_path = '/kaggle/working/MMTD/DATA/email_data/EDP.csv'
pics_path = '/kaggle/working/MMTD/DATA/email_data/pics'

# Veri ayırma
split_data = SplitData(csv_path, 5)
train_df, test_df = split_data()

train_dataset = EDPDataset(pics_path, train_df)
test_dataset = EDPDataset(pics_path, test_df)

model = MMTD()

training_args = TrainingArguments(
    output_dir='./output/PhishGuard_Asymmetric',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=EDPCollator(),
    compute_metrics=metrics,
)

print("🚀 MUTLAK YOLLAR KONTROL EDİLDİ. EĞİTİM BAŞLIYOR...")
trainer.train()

In [ ]:
%%writefile main.py
from Email_dataset import EDPDataset, EDPCollator
from models import AsymmetricGatedBEiTMMTD 
from transformers import Trainer, TrainingArguments
from torch.optim import AdamW, lr_scheduler
from utils import metrics, SplitData, save_config, EvalMetrics
import wandb
import os

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
fold = 5
split_data = SplitData('DATA/email_data/EDP.csv', fold)

bert_checkpoint_path = './output/bert/checkpoints'
bert_folds = os.listdir(bert_checkpoint_path)
bert_checkpoints = list()
for _ in os.listdir(bert_checkpoint_path):
    bert_checkpoints.extend(os.listdir(os.path.join(bert_checkpoint_path, _)))

dit_checkpoint_path = './output/dit/checkpoints'
dit_folds = os.listdir(dit_checkpoint_path)
dit_checkpoints = list()
for _ in os.listdir(dit_checkpoint_path):
    dit_checkpoints.extend(os.listdir(os.path.join(dit_checkpoint_path, _)))

if __name__ == '__main__':
    for i in range(fold):
        wandb.init(project='Asymmetric-Gated-MMTD')
        wandb.run.name = 'AsymGated-fold-' + str(i + 1)
        train_df, test_df = split_data()
        train_dataset = EDPDataset('DATA/email_data/pics', train_df)
        test_dataset = EDPDataset('DATA/email_data/pics', test_df)

        bert_checkpoint = os.path.join(bert_checkpoint_path, bert_folds[i], bert_checkpoints[i])
        dit_checkpoint = os.path.join(dit_checkpoint_path, dit_folds[i], dit_checkpoints[i])
        
        # YENİ MODELİMİZİ BAŞLATIYORUZ
        model = AsymmetricGatedBEiTMMTD(bert_pretrain_weight=bert_checkpoint, beit_pretrain_weight=dit_checkpoint)

        # Temel modelleri donduruyoruz (Sadece yeni eklediğimiz katmanlar eğitilecek)
        for p in model.text_encoder.parameters():
            p.requires_grad = False
        for p in model.image_encoder.parameters():
            p.requires_grad = False

        filtered_parameters = []
        for p in filter(lambda p: p.requires_grad, model.parameters()):
            filtered_parameters.append(p)

        optimizer = AdamW(filtered_parameters, lr=5e-4)

        args = TrainingArguments(
            output_dir='./output/AsymMMTD/checkpoints/fold' + str(i + 1),
            logging_dir='./output/AsymMMTD/log',
            logging_strategy='epoch',
            learning_rate=5e-4,
            per_device_train_batch_size=40,
            per_device_eval_batch_size=40,
            num_train_epochs=1, # DİKKAT: Bellek (OOM) testi için 1 yapıldı. Sorun çıkmazsa 3 yapabilirsin.
            weight_decay=0.0,
            save_strategy="epoch",
            evaluation_strategy="epoch",
            load_best_model_at_end=True,
            dataloader_num_workers=0,
            dataloader_pin_memory=True,
            run_name=wandb.run.name,
            auto_find_batch_size=False,
            overwrite_output_dir=True,
            save_total_limit=5,
            remove_unused_columns=False,
            report_to=["wandb"],
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            optimizers=(optimizer, None),
            data_collator=EDPCollator(),
            compute_metrics=metrics,
        )

        # Eğitimi başlatıyoruz (Çift yazım hatası düzeltildi)
        trainer.train()

        train_acc = trainer.evaluate(eval_dataset=train_dataset)
        train_result = {'train_acc': train_acc['eval_acc'], 'train_loss': train_acc['eval_loss']}
        wandb.log(train_result)

        trainer.compute_metrics = EvalMetrics('./output/AsymMMTD/results', wandb.run.name, True)
        test_acc = trainer.evaluate(eval_dataset=test_dataset)
        test_result = {'test_acc': test_acc['eval_acc'], 'test_loss': test_acc['eval_loss']}
        wandb.log(test_result)

        wandb.config = args.to_dict()
        save_config(args.to_dict(), os.path.join('./output/AsymMMTD/configs', wandb.run.name + '.yaml'))
        wandb.finish()

In [ ]:
!sed -i 's/BeitFeatureExtractor/BeitImageProcessor/g' Email_dataset.py

In [ ]:
!sed -i 's/CLIPFeatureExtractor/CLIPImageProcessor/g' Email_dataset.py
!sed -i 's/ViltFeatureExtractor/ViltImageProcessor/g' Email_dataset.py

In [ ]:
!sed -i 's/CLIPFeatureExtractor/CLIPImageProcessor/g' utils.py
!sed -i 's/ViltFeatureExtractor/ViltImageProcessor/g' utils.py
!sed -i 's/BeitFeatureExtractor/BeitImageProcessor/g' utils.py

In [ ]:
# utils.py dosyasını okuyoruz
with open('utils.py', 'r') as f:
    content = f.read()

# GloVe importunu zararsız bir boş sınıfla (mock) değiştiriyoruz
mock_glove = """
class GloVe:
    def __init__(self, *args, **kwargs):
        pass
"""
content = content.replace('from torchtext.vocab import GloVe', mock_glove)

# Dosyayı güncellenmiş haliyle geri yazıyoruz
with open('utils.py', 'w') as f:
    f.write(content)

print("✅ utils.py başarıyla yamalandı! torchtext belasından kurtulduk.")

In [ ]:
import re

# utils.py dosyasını okuyoruz
with open('utils.py', 'r') as f:
    content = f.read()

# İçinde 'torchtext' geçen tüm satırları siliyoruz
content = re.sub(r'^.*torchtext.*$', '', content, flags=re.MULTILINE)

# Eski kodların (bizim kullanmadığımız ama dosyada duran LSTM vb.) hata vermemesi için sahte fonksiyonlar
mocks = """
# --- MOCK TORCHTEXT KÖPRÜSÜ ---
class GloVe:
    def __init__(self, *args, **kwargs): pass
def get_tokenizer(*args, **kwargs): return lambda x: x
def build_vocab_from_iterator(*args, **kwargs): pass
# ------------------------------
"""

# Dosyayı temizlenmiş ve yamalanmış haliyle geri yazıyoruz
with open('utils.py', 'w') as f:
    f.write(mocks + "\n" + content)

print("✅ utils.py KAPSAMLI olarak temizlendi! Tüm torchtext bağları koparıldı.")

In [ ]:
import re

# Email_dataset.py dosyasını okuyoruz
with open('Email_dataset.py', 'r') as f:
    content = f.read()

# İçinde 'torchtext' geçen tüm satırları bulup siliyoruz
content = re.sub(r'^.*torchtext.*$', '', content, flags=re.MULTILINE)

# Eski kodların çökmesini engellemek için sahte sınıfları en başa ekliyoruz
mocks = """
# --- MOCK TORCHTEXT KÖPRÜSÜ ---
class GloVe:
    def __init__(self, *args, **kwargs): pass
def get_tokenizer(*args, **kwargs): return lambda x: x
# ------------------------------
"""

# Dosyayı tertemiz haliyle geri yazıyoruz
with open('Email_dataset.py', 'w') as f:
    f.write(mocks + "\n" + content)

print("✅ Email_dataset.py de başarıyla temizlendi! Artık karşımıza çıkamaz.")

In [ ]:
%cd /kaggle/working/MMTD/DATA


In [ ]:
!pip install -U gdown

In [ ]:
!gdown 1w4HxNf099lQ41mhMyXzbGwI429qBGvbY

In [ ]:
!unzip /kaggle/working/MMTD/DATA/email_data.zip -d /kaggle/working/MMTD/DATA/

In [ ]:
%%writefile /kaggle/working/MMTD/Email_dataset.py
from transformers import BertTokenizerFast, BeitImageProcessor
import pandas as pd
import torch
import os
from PIL import Image
from torch.utils.data import DataLoader

# ------------------------------
# DATASET
# ------------------------------

class EDPDataset(torch.utils.data.Dataset):
    def __init__(self, data_path, data_df):
        super().__init__()
        self.data_path = data_path
        self.data = data_df.reset_index(drop=True)

    def __getitem__(self, item):
        text = str(self.data.iloc[item, 0])
        pic_path = os.path.join(self.data_path, self.data.iloc[item, 1])

        # 🔥 LABEL FIX (EN KRİTİK)
        label = self.data.iloc[item, 2]
        label = int(label)
        label = 1 if label > 0 else 0

        image = Image.open(pic_path).convert("RGB")

        return text, image, label

    def __len__(self):
        return len(self.data)


# ------------------------------
# COLLATOR
# ------------------------------

class EDPCollator:
    def __init__(self):
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.processor = BeitImageProcessor.from_pretrained('microsoft/beit-base-patch16-224')

    def __call__(self, batch):
        texts, images, labels = zip(*batch)

        text_inputs = self.tokenizer(
            list(texts),
            return_tensors='pt',
            max_length=256,
            truncation=True,
            padding='max_length'
        )

        image_inputs = self.processor(
            list(images),
            return_tensors='pt'
        )

        # 🔥 LABEL FIX
        labels = torch.tensor(labels, dtype=torch.long)

        inputs = {}
        inputs.update(text_inputs)
        inputs.update(image_inputs)
        inputs["labels"] = labels

        return inputs


# ------------------------------
# TEST
# ------------------------------

if __name__ == "__main__":
    from utils import SplitData

    csv_path = 'DATA/email_data/EDP.csv'
    pics_path = 'DATA/email_data/pics'

    split = SplitData(csv_path, 5)
    train_df, _ = split()

    dataset = EDPDataset(pics_path, train_df)
    collator = EDPCollator()

    loader = DataLoader(dataset, batch_size=4, collate_fn=collator)

    batch = next(iter(loader))

    for k, v in batch.items():
        print(k, v.shape)

    print("LABELS:", batch["labels"])
    print("MIN:", batch["labels"].min().item(), "MAX:", batch["labels"].max().item())

In [ ]:
cd /kaggle/working/MMTD


In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import pandas as pd
from models import MMTD
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments
from utils import metrics

if __name__ == "__main__":
    df = pd.read_csv('DATA/email_data/EDP.csv')
    train_df = df.sample(frac=0.8, random_state=42)
    test_df = df.drop(train_df.index)

    model = MMTD()
    
    # 🔥 KAYIT AYARLARI
    args = TrainingArguments(
        output_dir='/kaggle/working/results',
        num_train_epochs=3, 
        per_device_train_batch_size=4,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_acc",
        save_total_limit=2,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=EDPDataset('DATA/email_data/pics', train_df),
        eval_dataset=EDPDataset('DATA/email_data/pics', test_df),
        data_collator=EDPCollator(),
        compute_metrics=metrics
    )
    
    print("🚀 DiT, Bidirectional Attention ve Modality Dropout ile eğitim başlıyor...")
    trainer.train()
    trainer.save_model('/kaggle/working/MMTD_Final_Model')

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
from transformers import BertModel, ViTModel
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss

class MMTD(nn.Module):
    def __init__(self, num_labels=2):
        super(MMTD, self).__init__()
        
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("microsoft/dit-base")
        
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        
        self.hidden_size = 768
        
        self.text_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Linear(self.hidden_size // 2, 1),
            nn.Sigmoid()
        )
        
        self.image_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Linear(self.hidden_size // 2, 1),
            nn.Sigmoid()
        )
        
        self.t2i_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        self.i2t_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        self.gate_t2i = nn.Sequential(nn.Linear(self.hidden_size * 2, self.hidden_size), nn.Sigmoid())
        self.gate_i2t = nn.Sequential(nn.Linear(self.hidden_size * 2, self.hidden_size), nn.Sigmoid())
        
        self.layer_norm = nn.LayerNorm(self.hidden_size)
        self.pooler = nn.Sequential(nn.Linear(self.hidden_size, self.hidden_size), nn.Tanh())
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, modality_dropout_prob=0.1, **kwargs):
        input_ids = input_ids.clamp(0, 30521)
        
        text_last = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        image_last = self.image_encoder(pixel_values=pixel_values).last_hidden_state

        if self.training:
            rand_val = torch.rand(1).item()
            if rand_val < modality_dropout_prob:
                image_last = torch.zeros_like(image_last)
            elif rand_val < 2 * modality_dropout_prob:
                text_last = torch.zeros_like(text_last)

        t_conf = self.text_confidence(text_last)
        i_conf = self.image_confidence(image_last)

        text_confident = text_last * t_conf
        image_confident = image_last * i_conf

        attn_t2i, _ = self.t2i_attention(query=text_last, key=image_confident, value=image_confident)
        gate_t = self.gate_t2i(torch.cat([text_last, attn_t2i], dim=-1))
        fused_text = text_last + (gate_t * attn_t2i)
        
        attn_i2t, _ = self.i2t_attention(query=image_last, key=text_confident, value=text_confident)
        gate_i = self.gate_i2t(torch.cat([image_last, attn_i2t], dim=-1))
        fused_image = image_last + (gate_i * attn_i2t)

        # 🔥 FİX: Dinamik Boyut Eşitleme (Hangisi kısaysa ona göre kes)
        min_len = min(fused_text.size(1), fused_image.size(1))
        
        combined = self.layer_norm(fused_text[:, :min_len, :] + fused_image[:, :min_len, :])
        
        # Sınıflandırma için [CLS] tokenını (0. indeks) alıyoruz, bu yüzden kırpma işlemi hiçbir veri kaybına yol açmaz
        pooled = self.pooler(combined[:, 0, :])
        logits = self.classifier(self.dropout(pooled))

        loss = None
        if labels is not None:
            labels = labels.clamp(0, 1).long()
            loss = CrossEntropyLoss()(logits.view(-1, 2), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/Email_dataset.py
from transformers import BertTokenizerFast, BeitImageProcessor
import pandas as pd
import torch
import os
from PIL import Image
from torch.utils.data import DataLoader

# ------------------------------
# DATASET
# ------------------------------

class EDPDataset(torch.utils.data.Dataset):
    def __init__(self, data_path, data_df):
        super().__init__()
        self.data_path = data_path
        self.data = data_df.reset_index(drop=True)

    def __getitem__(self, item):
        text = str(self.data.iloc[item, 0])
        pic_path = os.path.join(self.data_path, self.data.iloc[item, 1])

        label = self.data.iloc[item, 2]
        label = int(label)
        label = 1 if label > 0 else 0

        # 🔥 FİX: BOZUK/KAYIP DOSYA KORUMASI (Fault Tolerance)
        try:
            # Önce resmi normal şekilde açmayı dener
            image = Image.open(pic_path).convert("RGB")
        except (FileNotFoundError, IOError, OSError, Exception) as e:
            # Eğer resim yoksa, adı bozuksa veya inik değilse çökmek yerine boş (siyah) bir resim üretir.
            # 224x224 standart ViT/DiT giriş boyutudur.
            print(f"⚠️ Uyarı: Okunamayan resim atlandı. Yol: {pic_path} (Hata: {e})")
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        return text, image, label

    def __len__(self):
        return len(self.data)


# ------------------------------
# COLLATOR
# ------------------------------

class EDPCollator:
    def __init__(self):
        # DİKKAT: DiT mimarisi için BeitImageProcessor kullanılması harika bir uyum!
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.processor = BeitImageProcessor.from_pretrained('microsoft/beit-base-patch16-224')

    def __call__(self, batch):
        texts, images, labels = zip(*batch)

        text_inputs = self.tokenizer(
            list(texts),
            return_tensors='pt',
            max_length=256,
            truncation=True,
            padding='max_length'
        )

        image_inputs = self.processor(
            list(images),
            return_tensors='pt'
        )

        labels = torch.tensor(labels, dtype=torch.long)

        inputs = {}
        inputs.update(text_inputs)
        inputs.update(image_inputs)
        inputs["labels"] = labels

        return inputs


# ------------------------------
# TEST
# ------------------------------

if __name__ == "__main__":
    pass # Asıl test ve çalışma main.py üzerinden yapılacağı için bu kısmı geçebiliriz.

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import pandas as pd
from models import MMTD
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from utils import metrics

if __name__ == "__main__":
    # 1. Veri setini yükleme ve %80 Eğitim / %20 Test olarak bölme
    df = pd.read_csv('DATA/email_data/EDP.csv')
    train_df = df.sample(frac=0.8, random_state=42)
    test_df = df.drop(train_df.index)

    # 2. Modeli başlatma (Confidence Gating içeren o efsanevi hali)
    model = MMTD()
    
    # 🔥 3. ŞAMPİYONLAR LİGİ EĞİTİM AYARLARI
    args = TrainingArguments(
        output_dir='/kaggle/working/results',
        num_train_epochs=15,              # Uzun maraton hedefi
        per_device_train_batch_size=4,
        eval_strategy="epoch",            # Her epoch sonu başarıyı ölç
        save_strategy="epoch",            # Her epoch sonu ağırlıkları kaydet
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,      # Diskteki en İYİ modeli geri yükle
        metric_for_best_model="eval_acc", # Kriterimiz Accuracy (Doğruluk)
        greater_is_better=True,
        save_total_limit=2,               # Diski patlatmamak için sadece en iyi 2 checkpoint'i tut
        report_to="none"
    )

    # 4. Trainer Motorunu Kurma
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=EDPDataset('DATA/email_data/pics', train_df),
        eval_dataset=EDPDataset('DATA/email_data/pics', test_df),
        data_collator=EDPCollator(),
        compute_metrics=metrics,
        # 🔥 İŞTE O SİHİR: Erken Durdurma (Early Stopping)
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
    
    print("🚀 Öz-Farkındalıklı (Confidence Gating) Şampiyon Model Eğitimi Başlıyor...")
    print("🕒 Hedef: 15 Epoch (Erken durdurma 3 epoch sabırla aktif)")
    
    # 5. Eğitimi Başlat
    trainer.train()
    
    # 6. Zirvedeki Modeli Güvenli Klasöre Kaydet
    trainer.save_model('/kaggle/working/MMTD_Final_Champion_Model')
    print("✅ Eğitim tamamlandı ve şampiyon model diske başarıyla kaydedildi!")

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
from transformers import BertModel, ViTModel
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss


class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2, dropout_prob=0.1):
        super().__init__()

        # === ENCODERS ===
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("microsoft/dit-base")

        self.hidden_size = 768

        # === PARTIAL FREEZE (daha sağlıklı) ===
        for name, param in self.text_encoder.named_parameters():
            if "encoder.layer.10" not in name and "encoder.layer.11" not in name:
                param.requires_grad = False

        # === PROJECTION (ALIGNMENT FIX) ===
        self.text_proj = nn.Linear(self.hidden_size, self.hidden_size)
        self.image_proj = nn.Linear(self.hidden_size, self.hidden_size)

        # === CONFIDENCE ESTIMATORS ===
        self.text_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.hidden_size // 2, 1),
            nn.Sigmoid()
        )

        self.image_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.hidden_size // 2, 1),
            nn.Sigmoid()
        )

        # === CROSS ATTENTION ===
        self.t2i_attn = nn.MultiheadAttention(self.hidden_size, 8, batch_first=True)
        self.i2t_attn = nn.MultiheadAttention(self.hidden_size, 8, batch_first=True)

        # === ADVANCED GATING ===
        self.gate_t2i = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Sigmoid()
        )

        self.gate_i2t = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Sigmoid()
        )

        # === NORMALIZATION ===
        self.norm_t = nn.LayerNorm(self.hidden_size)
        self.norm_i = nn.LayerNorm(self.hidden_size)

        self.final_norm = nn.LayerNorm(self.hidden_size)

        # === CLASSIFIER ===
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        labels=None,
        modality_dropout_prob=0.1
    ):

        # === ENCODING ===
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        image_out = self.image_encoder(pixel_values=pixel_values)

        text_feat = self.text_proj(text_out.last_hidden_state)
        image_feat = self.image_proj(image_out.last_hidden_state)

        B = text_feat.size(0)

        # === SAMPLE-LEVEL MODALITY DROPOUT ===
        if self.training:
            drop_mask = torch.rand(B, device=text_feat.device)

            text_drop = drop_mask < modality_dropout_prob
            image_drop = (drop_mask >= modality_dropout_prob) & (drop_mask < 2 * modality_dropout_prob)

            text_feat[text_drop] = 0
            image_feat[image_drop] = 0

        # === CONFIDENCE ===
        t_conf = self.text_confidence(text_feat)  # [B, T, 1]
        i_conf = self.image_confidence(image_feat)

        # === ATTENTION BIAS (CRITICAL UPGRADE) ===
        # düşük confidence -> daha az attention
        t_weight = t_conf / (t_conf.sum(dim=1, keepdim=True) + 1e-6)
        i_weight = i_conf / (i_conf.sum(dim=1, keepdim=True) + 1e-6)

        text_weighted = text_feat * t_weight
        image_weighted = image_feat * i_weight

        # === CROSS ATTENTION ===
        t2i, _ = self.t2i_attn(
            query=text_feat,
            key=image_weighted,
            value=image_weighted
        )

        i2t, _ = self.i2t_attn(
            query=image_feat,
            key=text_weighted,
            value=text_weighted
        )

        # === GATED RESIDUAL FUSION ===
        gate_t = self.gate_t2i(torch.cat([text_feat, t2i], dim=-1))
        gate_i = self.gate_i2t(torch.cat([image_feat, i2t], dim=-1))

        fused_text = self.norm_t(text_feat + gate_t * t2i)
        fused_image = self.norm_i(image_feat + gate_i * i2t)

        # === POOLING (NO MORE HACK) ===
        text_cls = fused_text[:, 0, :]
        text_mean = fused_text.mean(dim=1)

        image_cls = fused_image[:, 0, :]
        image_mean = fused_image.mean(dim=1)

        text_repr = 0.7 * text_cls + 0.3 * text_mean
        image_repr = 0.7 * image_cls + 0.3 * image_mean

        combined = self.final_norm(text_repr + image_repr)

        logits = self.classifier(self.dropout(combined))

        # === LOSS ===
        loss = None
        if labels is not None:
            loss = CrossEntropyLoss()(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
from transformers import BertModel, ViTModel
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss


class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2, dropout_prob=0.1):
        super().__init__()

        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("microsoft/dit-base")

        self.hidden_size = 768

        # Partial freeze (son 2 layer trainable)
        for name, param in self.text_encoder.named_parameters():
            if "encoder.layer.10" not in name and "encoder.layer.11" not in name:
                param.requires_grad = False

        # Projection
        self.text_proj = nn.Linear(self.hidden_size, self.hidden_size)
        self.image_proj = nn.Linear(self.hidden_size, self.hidden_size)

        # Confidence
        self.text_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.hidden_size // 2, 1)
        )

        self.image_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.hidden_size // 2, 1)
        )

        # Cross attention
        self.t2i_attn = nn.MultiheadAttention(self.hidden_size, 8, batch_first=True)
        self.i2t_attn = nn.MultiheadAttention(self.hidden_size, 8, batch_first=True)

        # Gating
        self.gate_t2i = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Sigmoid()
        )

        self.gate_i2t = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Sigmoid()
        )

        self.norm_t = nn.LayerNorm(self.hidden_size)
        self.norm_i = nn.LayerNorm(self.hidden_size)
        self.final_norm = nn.LayerNorm(self.hidden_size)

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        labels=None,
        modality_dropout_prob=0.1
    ):

        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        image_out = self.image_encoder(pixel_values=pixel_values)

        text_feat = self.text_proj(text_out.last_hidden_state)
        image_feat = self.image_proj(image_out.last_hidden_state)

        B = text_feat.size(0)

        # Sample-level modality dropout (safe version)
        if self.training:
            drop_mask = torch.rand(B, device=text_feat.device)

            text_drop = drop_mask < modality_dropout_prob
            image_drop = (drop_mask >= modality_dropout_prob) & (drop_mask < 2 * modality_dropout_prob)

            text_feat = text_feat.masked_fill(text_drop.view(-1,1,1), 0)
            image_feat = image_feat.masked_fill(image_drop.view(-1,1,1), 0)

        # Confidence → softmax (stable)
        t_conf = torch.softmax(self.text_confidence(text_feat).squeeze(-1), dim=1).unsqueeze(-1)
        i_conf = torch.softmax(self.image_confidence(image_feat).squeeze(-1), dim=1).unsqueeze(-1)

        text_weighted = text_feat * t_conf
        image_weighted = image_feat * i_conf

        # Cross attention
        t2i, _ = self.t2i_attn(query=text_feat, key=image_weighted, value=image_weighted)
        i2t, _ = self.i2t_attn(query=image_feat, key=text_weighted, value=text_weighted)

        # Fusion
        gate_t = self.gate_t2i(torch.cat([text_feat, t2i], dim=-1))
        gate_i = self.gate_i2t(torch.cat([image_feat, i2t], dim=-1))

        fused_text = self.norm_t(text_feat + gate_t * t2i)
        fused_image = self.norm_i(image_feat + gate_i * i2t)

        # Pooling
        text_repr = 0.7 * fused_text[:, 0, :] + 0.3 * fused_text.mean(dim=1)
        image_repr = 0.7 * fused_image[:, 0, :] + 0.3 * fused_image.mean(dim=1)

        combined = self.final_norm(text_repr + image_repr)

        logits = self.classifier(self.dropout(combined))

        loss = None
        if labels is not None:
            loss = CrossEntropyLoss()(logits, labels.view(-1).long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import pandas as pd
from models import MMTD_Advanced
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from utils import metrics
from sklearn.model_selection import train_test_split


if __name__ == "__main__":

    # === DATA ===
    df = pd.read_csv('DATA/email_data/EDP.csv')

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    # === MODEL ===
    model = MMTD_Advanced()

    # === TRAINING ARGS ===
    args = TrainingArguments(
        output_dir='/kaggle/working/results',

        num_train_epochs=15,
        per_device_train_batch_size=4,

        gradient_accumulation_steps=4,

        eval_strategy="epoch",
        save_strategy="epoch",

        logging_strategy="steps",
        logging_steps=50,
        logging_dir='/kaggle/working/logs',

        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,

        save_total_limit=2,

        learning_rate=1e-5,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        weight_decay=0.01,

        fp16=True,

        remove_unused_columns=False,

        report_to="none"
    )

    # === TRAINER ===
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=EDPDataset('DATA/email_data/pics', train_df),
        eval_dataset=EDPDataset('DATA/email_data/pics', test_df),
        data_collator=EDPCollator(),
        compute_metrics=metrics,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001
        )]
    )

    print("🚀 Advanced Multimodal Model Training Started")

    trainer.train()

    trainer.save_model('/kaggle/working/MMTD_Final_Model')

    print("✅ Training completed and model saved!")

In [ ]:
import pandas as pd

df = pd.read_csv('DATA/email_data/EDP.csv')

print(df.columns)
print(df.head())

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import pandas as pd
from models import MMTD_Advanced
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from utils import metrics
from sklearn.model_selection import train_test_split


if __name__ == "__main__":

    # === DATA ===
    df = pd.read_csv('DATA/email_data/EDP.csv')

    # 🔥 KOLON DÜZELTME
    df = df.rename(columns={
        "texts": "text",
        "pics": "image",
        "labels": "label"
    })

    print("Columns:", df.columns)

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    # === MODEL ===
    model = MMTD_Advanced()

    # === TRAINING ARGS ===
    args = TrainingArguments(
        output_dir='/kaggle/working/results',
        num_train_epochs=15,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,

        eval_strategy="epoch",
        save_strategy="epoch",

        logging_strategy="steps",
        logging_steps=50,
        logging_dir='/kaggle/working/logs',

        load_best_model_at_end=True,
        metric_for_best_model="eval_acc",
        greater_is_better=True,

        save_total_limit=2,

        learning_rate=1e-5,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        weight_decay=0.01,

        fp16=True,
        remove_unused_columns=False,

        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=EDPDataset('DATA/email_data/pics', train_df),
        eval_dataset=EDPDataset('DATA/email_data/pics', test_df),
        data_collator=EDPCollator(),
        compute_metrics=metrics,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001
        )]
    )

    print("🚀 Training started...")

    trainer.train()

    trainer.save_model('/kaggle/working/MMTD_Final_Model')

    print("✅ Training completed!")

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
from transformers import BertModel, ViTModel
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss


class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2, dropout_prob=0.1):
        super().__init__()

        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("microsoft/dit-base")

        self.hidden_size = 768

        # Partial freeze (son 2 layer trainable)
        for name, param in self.text_encoder.named_parameters():
            if "encoder.layer.10" not in name and "encoder.layer.11" not in name:
                param.requires_grad = False

        # Projection
        self.text_proj = nn.Linear(self.hidden_size, self.hidden_size)
        self.image_proj = nn.Linear(self.hidden_size, self.hidden_size)

        # Confidence
        self.text_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.hidden_size // 2, 1)
        )

        self.image_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.hidden_size // 2, 1)
        )

        # Cross attention
        self.t2i_attn = nn.MultiheadAttention(self.hidden_size, 8, batch_first=True)
        self.i2t_attn = nn.MultiheadAttention(self.hidden_size, 8, batch_first=True)

        # Gating
        self.gate_t2i = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Sigmoid()
        )

        self.gate_i2t = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Sigmoid()
        )

        self.norm_t = nn.LayerNorm(self.hidden_size)
        self.norm_i = nn.LayerNorm(self.hidden_size)
        self.final_norm = nn.LayerNorm(self.hidden_size)

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    def forward(
    self,
    input_ids,
    attention_mask,
    pixel_values,
    token_type_ids=None,   # 🔥 BUNU EKLE
    labels=None,
    modality_dropout_prob=0.1,
    **kwargs               # 🔥 BUNU DA EKLE
):

        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        image_out = self.image_encoder(pixel_values=pixel_values)

        text_feat = self.text_proj(text_out.last_hidden_state)
        image_feat = self.image_proj(image_out.last_hidden_state)

        B = text_feat.size(0)

        # Sample-level modality dropout (safe version)
        if self.training:
            drop_mask = torch.rand(B, device=text_feat.device)

            text_drop = drop_mask < modality_dropout_prob
            image_drop = (drop_mask >= modality_dropout_prob) & (drop_mask < 2 * modality_dropout_prob)

            text_feat = text_feat.masked_fill(text_drop.view(-1,1,1), 0)
            image_feat = image_feat.masked_fill(image_drop.view(-1,1,1), 0)

        # Confidence → softmax (stable)
        t_conf = torch.softmax(self.text_confidence(text_feat).squeeze(-1), dim=1).unsqueeze(-1)
        i_conf = torch.softmax(self.image_confidence(image_feat).squeeze(-1), dim=1).unsqueeze(-1)

        text_weighted = text_feat * t_conf
        image_weighted = image_feat * i_conf

        # Cross attention
        t2i, _ = self.t2i_attn(query=text_feat, key=image_weighted, value=image_weighted)
        i2t, _ = self.i2t_attn(query=image_feat, key=text_weighted, value=text_weighted)

        # Fusion
        gate_t = self.gate_t2i(torch.cat([text_feat, t2i], dim=-1))
        gate_i = self.gate_i2t(torch.cat([image_feat, i2t], dim=-1))

        fused_text = self.norm_t(text_feat + gate_t * t2i)
        fused_image = self.norm_i(image_feat + gate_i * i2t)

        # Pooling
        text_repr = 0.7 * fused_text[:, 0, :] + 0.3 * fused_text.mean(dim=1)
        image_repr = 0.7 * fused_image[:, 0, :] + 0.3 * fused_image.mean(dim=1)

        combined = self.final_norm(text_repr + image_repr)

        logits = self.classifier(self.dropout(combined))

        loss = None
        if labels is not None:
            loss = CrossEntropyLoss()(logits, labels.view(-1).long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
from transformers import BertModel, ViTModel
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss


class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2, dropout_prob=0.1):
        super().__init__()

        
        self.text_encoder = BertModel.from_pretrained("bert-base-multilingual-cased")
        self.image_encoder = ViTModel.from_pretrained("microsoft/dit-base")

        self.hidden_size = 768

        # Partial freeze (son 2 layer trainable)
        for name, param in self.text_encoder.named_parameters():
            if "encoder.layer.10" not in name and "encoder.layer.11" not in name:
                param.requires_grad = False

        # Projection
        self.text_proj = nn.Linear(self.hidden_size, self.hidden_size)
        self.image_proj = nn.Linear(self.hidden_size, self.hidden_size)

        # Confidence
        self.text_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.hidden_size // 2, 1)
        )

        self.image_confidence = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.hidden_size // 2, 1)
        )

        # Cross attention
        self.t2i_attn = nn.MultiheadAttention(self.hidden_size, 8, batch_first=True)
        self.i2t_attn = nn.MultiheadAttention(self.hidden_size, 8, batch_first=True)

        # Gating
        self.gate_t2i = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Sigmoid()
        )

        self.gate_i2t = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Sigmoid()
        )

        self.norm_t = nn.LayerNorm(self.hidden_size)
        self.norm_i = nn.LayerNorm(self.hidden_size)
        self.final_norm = nn.LayerNorm(self.hidden_size)

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    def forward(
    self,
    input_ids,
    attention_mask,
    pixel_values,
    token_type_ids=None,   # 🔥 BUNU EKLE
    labels=None,
    modality_dropout_prob=0.1,
    **kwargs               # 🔥 BUNU DA EKLE
):

        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        image_out = self.image_encoder(pixel_values=pixel_values)

        text_feat = self.text_proj(text_out.last_hidden_state)
        image_feat = self.image_proj(image_out.last_hidden_state)

        B = text_feat.size(0)

        # Sample-level modality dropout (safe version)
        if self.training:
            drop_mask = torch.rand(B, device=text_feat.device)

            text_drop = drop_mask < modality_dropout_prob
            image_drop = (drop_mask >= modality_dropout_prob) & (drop_mask < 2 * modality_dropout_prob)

            text_feat = text_feat.masked_fill(text_drop.view(-1,1,1), 0)
            image_feat = image_feat.masked_fill(image_drop.view(-1,1,1), 0)

        # Confidence → softmax (stable)
        t_conf = torch.softmax(self.text_confidence(text_feat).squeeze(-1), dim=1).unsqueeze(-1)
        i_conf = torch.softmax(self.image_confidence(image_feat).squeeze(-1), dim=1).unsqueeze(-1)

        text_weighted = text_feat * t_conf
        image_weighted = image_feat * i_conf

        # Cross attention
        t2i, _ = self.t2i_attn(query=text_feat, key=image_weighted, value=image_weighted)
        i2t, _ = self.i2t_attn(query=image_feat, key=text_weighted, value=text_weighted)

        # Fusion
        gate_t = self.gate_t2i(torch.cat([text_feat, t2i], dim=-1))
        gate_i = self.gate_i2t(torch.cat([image_feat, i2t], dim=-1))

        fused_text = self.norm_t(text_feat + gate_t * t2i)
        fused_image = self.norm_i(image_feat + gate_i * i2t)

        # Pooling
        text_repr = 0.7 * fused_text[:, 0, :] + 0.3 * fused_text.mean(dim=1)
        image_repr = 0.7 * fused_image[:, 0, :] + 0.3 * fused_image.mean(dim=1)

        combined = self.final_norm(text_repr + image_repr)

        logits = self.classifier(self.dropout(combined))

        loss = None
        if labels is not None:
            loss = CrossEntropyLoss()(logits, labels.view(-1).long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import pandas as pd
import matplotlib.pyplot as plt
from models import MMTD_Advanced
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
from utils import metrics
from sklearn.model_selection import train_test_split

# === CANLI GRAFİK ÇİZİCİ CALLBACK ===
class PlotMetricsCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.train_loss = []
        self.eval_loss = []
        self.eval_acc = []
        os.makedirs(self.output_dir, exist_ok=True)

    def on_epoch_end(self, args, state, control, logs=None, **kwargs):
        """Her epoch sonunda loglardan değerleri çeker ve grafik çizer."""
        # Son logları kontrol et (hem eğitim hem test logları aynı anda gelmeyebilir, bu yüzden geçmişi tarıyoruz)
        current_train_loss = None
        current_eval_loss = None
        current_eval_acc = None

        for hist in state.log_history:
            if 'loss' in hist and 'epoch' in hist and hist['epoch'] == round(state.epoch):
                current_train_loss = hist['loss']
            if 'eval_loss' in hist and 'epoch' in hist and hist['epoch'] == round(state.epoch):
                current_eval_loss = hist['eval_loss']
                current_eval_acc = hist.get('eval_acc', None)

        if current_train_loss is not None: self.train_loss.append(current_train_loss)
        if current_eval_loss is not None: self.eval_loss.append(current_eval_loss)
        if current_eval_acc is not None: self.eval_acc.append(current_eval_acc)

        self.plot_graphs(state.epoch)

    def plot_graphs(self, epoch):
        """Grafikleri çizip kaydeder."""
        plt.figure(figsize=(12, 5))

        # Kayıp (Loss) Grafiği
        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, label='Train Loss', marker='o', color='blue')
        if len(self.eval_loss) == len(self.train_loss):
            plt.plot(self.eval_loss, label='Eval Loss', marker='s', color='red')
        plt.title(f'Loss Curve (Epoch {epoch:.0f})')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        # Başarı (Accuracy) Grafiği
        plt.subplot(1, 2, 2)
        plt.plot(self.eval_acc, label='Eval Accuracy', marker='^', color='green')
        plt.title(f'Accuracy Curve (Epoch {epoch:.0f})')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        # Grafiği Kaggle klasörüne resim olarak kaydet (İnternet kopsa bile elinde kalır)
        save_path = os.path.join(self.output_dir, 'training_metrics_live.png')
        plt.savefig(save_path)
        
        # Ekranında görünmesi için
        plt.show(block=False)
        plt.pause(1) 
        plt.close()


if __name__ == "__main__":

    # === DATA ===
    df = pd.read_csv('DATA/email_data/EDP.csv')

    # 🔥 KOLON DÜZELTME
    df = df.rename(columns={
        "texts": "text",
        "pics": "image",
        "labels": "label"
    })

    print("Columns:", df.columns)

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    # === MODEL ===
    model = MMTD_Advanced()

    # === TRAINING ARGS ===
    args = TrainingArguments(
        output_dir='/kaggle/working/results',
        num_train_epochs=15,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,

        eval_strategy="epoch",  # eval_strategy güncel formattır (evaluation_strategy yerine)
        save_strategy="epoch",

        logging_strategy="epoch", # Grafikler epoch bazlı olduğu için logları da epoch yapıyoruz
        logging_dir='/kaggle/working/logs',

        load_best_model_at_end=True,
        metric_for_best_model="eval_acc", # ERKEN DURDURMA İÇİN KESİN DEĞER
        greater_is_better=True,

        save_total_limit=2,

        learning_rate=1e-5,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        weight_decay=0.01,

        fp16=True,
        remove_unused_columns=False,

        report_to="none" 
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=EDPDataset('DATA/email_data/pics', train_df),
        eval_dataset=EDPDataset('DATA/email_data/pics', test_df),
        data_collator=EDPCollator(),
        compute_metrics=metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=3, # 3 epoch boyunca eval_acc artmazsa dur!
                early_stopping_threshold=0.001
            ),
            PlotMetricsCallback(output_dir='/kaggle/working/results') # ÇİZİM CALLBACK'İ EKLENDİ
        ]
    )

    print("🚀 Training started (Early Stopping and Live Plotting Enabled)...")

    trainer.train()

    trainer.save_model('/kaggle/working/MMTD_Final_Model')

    print("✅ Training completed!")

In [ ]:
import sys
import os
import torch
import random
import pandas as pd
import numpy as np
from PIL import Image
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BeitImageProcessor
from sklearn.model_selection import train_test_split
from safetensors.torch import load_file

# Kendi mimari kodlarını Kaggle'a tanıtıyoruz
sys.path.append('/kaggle/working/MMTD')
from models import MMTD_Advanced
from Email_dataset import EDPDataset
from utils import metrics

# =====================================================================
# 🚨 %100 ORİJİNAL MAKALE KODLARI (Siber Saldırı Simülasyonu)
# =====================================================================
def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape).astype(np.uint8)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        index = random.randint(0, len(text) - 1)
        noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

# Kendi ayakları üzerinde duran, hatasız yeni Gürültü Yükleyici
class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        # Kütüphane ismini güncelledik (BeitImageProcessor oldu)
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')

    def __call__(self, data):
        text_list, picture_list, label_list = zip(*data)
        
        # Test anında havada gürültü (saldırı) ekle
        if self.text_noise_level > 0:
            text_list = [add_text_noise(t, self.text_noise_level) for t in text_list]
        if self.image_noise:
            picture_list = [add_gaussian_noise(p) for p in picture_list]

        # Bozulmuş verileri tensor'a çevir
        text_tensor = self.tokenizer(list(text_list), return_tensors='pt', max_length=256, truncation=True, padding='max_length')
        pixel_values = self.feature_extractor(list(picture_list), return_tensors='pt')
        labels = torch.LongTensor(label_list)
        
        inputs = dict()
        inputs.update(text_tensor)
        inputs.update(pixel_values)
        inputs['labels'] = labels
        return inputs

# =====================================================================
# 📊 TEST LABORATUVARI
# =====================================================================
if __name__ == "__main__":
    print("🚨 Orijinal Makale Şartlarında Gürbüzlük (Robustness) Testi Başlıyor...\n")

    # Veri seti yolları
    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    
    # 5 Fold Cross Validation mantığıyla aynı test setini ayırıyoruz
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    test_dataset = EDPDataset(RESIMLERIN_YOLU, test_df)

    # 🧠 SENİN ŞAMPİYON MODELİNİ YÜKLÜYORUZ
    print("🧠 Şampiyon model (MMTD_Advanced) uyandırılıyor...")
    model = MMTD_Advanced()
    
    # Kaggle Input Dataset yolu
    model_path = '/kaggle/input/datasets/arafkubraa/confidencemodelsafetors/model.safetensors'
    model.load_state_dict(load_file(model_path))
    model.eval() # Model değerlendirme modunda

    args = TrainingArguments(
        output_dir='./test_results',
        per_device_eval_batch_size=8,
        report_to="none"
    )

    # Orijinal makaledeki 4 saldırı senaryosu
    scenarios = [
        {"name": "1. Temiz Veri (Baseline)", "text_noise": 0.0, "image_noise": False},
        {"name": "2. Sadece Resim Gürültüsü (Gaussian)", "text_noise": 0.0, "image_noise": True},
        {"name": "3. Sadece Metin Gürültüsü (%10)", "text_noise": 0.1, "image_noise": False},
        {"name": "4. Tam Kaos (Metin %10 + Resim)", "text_noise": 0.1, "image_noise": True},
    ]

    print("\n📊 --- HAKEMLER İÇİN KARŞILAŞTIRMALI TEST SONUÇLARI ---")
    for s in scenarios:
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        
        trainer = Trainer(
            model=model,
            args=args,
            eval_dataset=test_dataset,
            data_collator=collator,
            compute_metrics=metrics
        )
        
        result = trainer.evaluate() 
        print(f"✅ [{s['name']}] -> Başarı (Accuracy): %{result['eval_acc']*100:.2f} | Kayıp (Loss): {result['eval_loss']:.4f}\n")

In [ ]:
 import sys
import os
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BeitImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from safetensors.torch import load_file

# Kendi mimari kodlarını Kaggle'a tanıtıyoruz
sys.path.append('/kaggle/working/MMTD')
from models import MMTD_Advanced
from Email_dataset import EDPDataset

# =====================================================================
# 🚨 1. GÜRÜLTÜ VE VERİ YÜKLEYİCİ
# =====================================================================
def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape).astype(np.uint8)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        index = random.randint(0, len(text) - 1)
        noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')

    def __call__(self, data):
        text_list, picture_list, label_list = zip(*data)
        
        if self.text_noise_level > 0:
            text_list = [add_text_noise(t, self.text_noise_level) for t in text_list]
        if self.image_noise:
            picture_list = [add_gaussian_noise(p) for p in picture_list]

        text_tensor = self.tokenizer(list(text_list), return_tensors='pt', max_length=256, truncation=True, padding='max_length')
        pixel_values = self.feature_extractor(list(picture_list), return_tensors='pt')
        labels = torch.LongTensor(label_list)
        
        inputs = dict()
        inputs.update(text_tensor)
        inputs.update(pixel_values)
        inputs['labels'] = labels
        return inputs

# =====================================================================
# 📊 2. GELİŞMİŞ METRİK HESAPLAYICI (Makale Standartlarında)
# =====================================================================
def compute_advanced_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    # Makro ortalama, dengesiz veri setlerinde en adil sonucu verir
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    
    return {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    }

# =====================================================================
# 🚀 3. TEST LABORATUVARI VE GRAFİK ÇİZİMİ
# =====================================================================
if __name__ == "__main__":
    print("🚨 Nihai Raporlama ve Çizim Laboratuvarı Başlıyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU)
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    test_dataset = EDPDataset(RESIMLERIN_YOLU, test_df)

    model = MMTD_Advanced()
    model_path = '/kaggle/input/datasets/arafkubraa/confidencemodelsafetors/model.safetensors'
    model.load_state_dict(load_file(model_path))
    model.eval()

    args = TrainingArguments(
        output_dir='./test_results',
        per_device_eval_batch_size=8,
        report_to="none"
    )

    scenarios = [
        {"name": "Temiz Veri", "text_noise": 0.0, "image_noise": False},
        {"name": "Görüntü Bozuk", "text_noise": 0.0, "image_noise": True},
        {"name": "Metin Bozuk", "text_noise": 0.1, "image_noise": False},
        {"name": "Tam Kaos", "text_noise": 0.1, "image_noise": True},
    ]

    # Sonuçları kaydedeceğimiz listeler
    names, accuracies, f1_scores = [], [], []

    print("\n📊 --- MMTD_ADVANCED DETAYLI SONUÇLAR ---")
    for s in scenarios:
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        trainer = Trainer(
            model=model,
            args=args,
            eval_dataset=test_dataset,
            data_collator=collator,
            compute_metrics=compute_advanced_metrics # Yeni fonksiyonumuz!
        )
        
        result = trainer.evaluate() 
        acc = result['eval_accuracy'] * 100
        f1 = result['eval_f1_score'] * 100
        
        names.append(s['name'])
        accuracies.append(acc)
        f1_scores.append(f1)
        
        print(f"✅ [{s['name']}] -> Acc: %{acc:.2f} | F1: %{f1:.2f} | Precision: %{result['eval_precision']*100:.2f} | Recall: %{result['eval_recall']*100:.2f}")

    # =====================================================================
    # 🎨 4. MAKALE İÇİN GRAFİK ÇİZİMİ
    # =====================================================================
    print("\n🎨 Makale için grafik çiziliyor...")
    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#2ca02c')
    rects2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#1f77b4')

    ax.set_ylabel('Yüzde (%)', fontsize=12, fontweight='bold')
    ax.set_title('MMTD_Advanced Modeli Gürbüzlük (Robustness) Performansı', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylim([0, 110])
    ax.legend(loc='lower left')

    # Barların üzerine değerleri yazdırma
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  # 3 points vertical offset
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)

    autolabel(rects1)
    autolabel(rects2)
    plt.tight_layout()
    
    # Resmi Kaggle working klasörüne kaydet
    plt.savefig('/kaggle/working/mmtd_advanced_robustness.png', dpi=300)
    plt.show()
    print("✅ Grafik '/kaggle/working/mmtd_advanced_robustness.png' olarak kaydedildi! İndirebilirsin.")

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, ViTModel
from transformers.modeling_outputs import SequenceClassifierOutput

class MMTD_Advanced(nn.Module): # Main.py ile uyumlu olması için ismi koruduk
    def __init__(self, num_labels=2, dropout_prob=0.2):
        super().__init__()

        # --- Encoders ---
        # Metin için standart BERT, Resim için Genel Vision Transformer (ViT)
        self.text_encoder = BertModel.from_pretrained("bert-base-multilingual-cased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        hidden = 768

        # --- Projection (Alignment Space) ---
        # Özellikleri ortak bir matematiksel uzaya taşır ve stabilize eder
        self.text_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU()
        )

        self.image_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU()
        )

        # --- Learnable Uncertainty Heads ---
        # Hangi token veya patch'in gürültülü olduğunu öğrenen kafa
        self.text_uncertainty = nn.Linear(hidden, 1)
        self.image_uncertainty = nn.Linear(hidden, 1)

        # --- Global Modality Prior ---
        # Hangi kanalın (metin/resim) genel olarak daha güvenilir olduğunu öğrenir
        self.modality_scale = nn.Parameter(torch.tensor([1.0, 1.0]))

        # --- Stochastic Fusion Gate ---
        # Füzyonun ne kadar derin olacağına karar veren dinamik kapı
        self.fusion_gate = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden, hidden),
            nn.Sigmoid()
        )

        # --- Fusion Network ---
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden, hidden)
        )

        self.classifier = nn.Linear(hidden, num_labels)
        self.dropout = nn.Dropout(dropout_prob)

    def masked_mean(self, x, mask=None):
        """Tüm patch/token'ların ortalamasını alarak stabil temsil oluşturur."""
        if mask is None:
            return x.mean(dim=1)
        mask = mask.unsqueeze(-1).float()
        return (x * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-6)

    def uncertainty_weight(self, feat, uncertainty_head):
        """Belirsizliği yüksek olan kısımları filtreler."""
        unc = torch.sigmoid(uncertainty_head(feat))  # [B, T, 1]
        weight = 1.0 - unc # Belirsizlik arttıkça ağırlık düşer
        return feat * weight

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, modality_dropout_prob=0.15, **kwargs):

        # --- 1. Encoding ---
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        image_out = self.image_encoder(pixel_values=pixel_values)

        text_feat = self.text_proj(text_out.last_hidden_state)
        image_feat = self.image_proj(image_out.last_hidden_state)

        # --- 2. Uncertainty-Aware Filtering ---
        # Gürültülü bölgeler (noise) burada süzülür
        text_feat = self.uncertainty_weight(text_feat, self.text_uncertainty)
        image_feat = self.uncertainty_weight(image_feat, self.image_uncertainty)

        # --- 3. Stochastic Modality Dropout ---
        if self.training:
            noise_t = torch.bernoulli(torch.full((1,), modality_dropout_prob, device=text_feat.device))
            noise_i = torch.bernoulli(torch.full((1,), modality_dropout_prob, device=image_feat.device))
            text_feat = text_feat * (1 - noise_t)
            image_feat = image_feat * (1 - noise_i)

        # --- 4. Stable Pooling ---
        text_repr = self.masked_mean(text_feat, attention_mask)
        image_repr = self.masked_mean(image_feat)

        # --- 5. Global Modality Weighting ---
        weights = F.softmax(self.modality_scale, dim=0)
        text_repr = text_repr * weights[0]
        image_repr = image_repr * weights[1]

        # --- 6. Stochastic Gated Fusion ---
        fused_input = torch.cat([text_repr, image_repr], dim=-1)
        gate = self.fusion_gate(fused_input)
        fused_transformed = self.fusion(fused_input)

        # Kapı kararına göre orijinal birleşim ile işlenmiş füzyon arasında denge kurar
        fused = fused_transformed * gate + fused_input * (1 - gate)

        logits = self.classifier(self.dropout(fused))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels.view(-1).long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from models import MMTD_Advanced
from transformers import (
    Trainer, TrainingArguments, EarlyStoppingCallback, 
    TrainerCallback, BertTokenizerFast, BeitImageProcessor
)
from utils import metrics
from sklearn.model_selection import train_test_split

# =====================================================================
# 🌪️ GÜRÜLTÜ FONKSİYONLARI (Helper Functions)
# =====================================================================
def add_gaussian_noise(image, std=25):
    img_array = np.array(image)
    noise = np.random.normal(0, std, img_array.shape).astype(np.uint8)
    noisy_img = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_img)

def add_text_noise(text, noise_level=0.1):
    if not text or noise_level <= 0: return text
    chars = list(text)
    num_noise = int(len(chars) * noise_level)
    for _ in range(num_noise):
        idx = random.randint(0, len(chars) - 1)
        chars[idx] = random.choice("abcdefghijklmnopqrstuvwxyz0123456789!@#$%^&*")
    return "".join(chars)

# =====================================================================
# 🛡️ DİNAMİK GÜRÜLTÜCÜ (Dynamic Robust Collator)
# =====================================================================
class DynamicRobustCollator:
    def __init__(self, max_epochs=15):
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = BeitImageProcessor.from_pretrained('microsoft/dit-base')
        self.current_epoch = 0
        self.max_epochs = max_epochs
        self.max_noise_prob = 0.6  # Maksimum %60 gürültü ihtimali

    def __call__(self, features):
        # Epoch ilerledikçe gürültü ihtimali artar (Linear Scheduling)
        noise_prob = (self.current_epoch / self.max_epochs) * self.max_noise_prob
        
        texts, images, labels = [], [], []
        for f in features:
            txt, img, lbl = f["text"], f["image"], f["label"]
            
            # Rastgele gürültü uygula
            if random.random() < noise_prob:
                txt = add_text_noise(txt)
                img = add_gaussian_noise(img)
            
            texts.append(txt)
            images.append(img)
            labels.append(lbl)

        inputs = self.tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
        pixel_values = self.feature_extractor(images, return_tensors="pt")["pixel_values"]
        
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels, dtype=torch.long)
        }

# =====================================================================
# 📉 CALLBACKS (Plotting & Noise Scheduling)
# =====================================================================
class NoiseSchedulerCallback(TrainerCallback):
    """Her epoch başında collator'daki epoch bilgisini günceller."""
    def on_epoch_begin(self, args, state, control, **kwargs):
        if hasattr(kwargs['train_dataloader'].collate_fn, 'current_epoch'):
            kwargs['train_dataloader'].collate_fn.current_epoch = state.epoch
            print(f"\n🔔 Epoch {state.epoch:.0f} - Mevcut Gürültü İhtimali: %{ (state.epoch/15)*60:.1f}")

class PlotMetricsCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.train_loss, self.eval_loss, self.eval_acc = [], [], []
        os.makedirs(self.output_dir, exist_ok=True)

    def on_epoch_end(self, args, state, control, logs=None, **kwargs):
        # Logları ayıkla
        for hist in state.log_history:
            if 'loss' in hist and hist['epoch'] == round(state.epoch):
                self.train_loss.append(hist['loss'])
            if 'eval_loss' in hist and hist['epoch'] == round(state.epoch):
                self.eval_loss.append(hist['eval_loss'])
                self.eval_acc.append(hist.get('eval_acc', 0))
        self.plot_graphs(state.epoch)

    def plot_graphs(self, epoch):
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, label='Train Loss', color='blue')
        if len(self.eval_loss) == len(self.train_loss):
            plt.plot(self.eval_loss, label='Eval Loss', color='red')
        plt.title(f'Loss (Epoch {epoch:.0f})'); plt.legend(); plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(self.eval_acc, label='Eval Accuracy', color='green')
        plt.title(f'Accuracy (Epoch {epoch:.0f})'); plt.legend(); plt.grid(True)

        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, 'training_metrics_live.png'))
        plt.show(block=False); plt.pause(0.1); plt.close()

# =====================================================================
# 🚀 MAIN EXECUTION
# =====================================================================
if __name__ == "__main__":
    # Veriyi yükle ve temizle
    df = pd.read_csv('DATA/email_data/EDP.csv')
    df = df.rename(columns={"texts": "text", "pics": "image", "labels": "label"})
    
    # NaN değerleri temizle (Hata almamak için kritik)
    df['text'] = df['text'].fillna("")
    
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

    # Dataset ve Collator
    from Email_dataset import EDPDataset # Orijinal dataset sınıfın
    train_dataset = EDPDataset('DATA/email_data/pics', train_df)
    test_dataset = EDPDataset('DATA/email_data/pics', test_df)
    
    # ⚠️ Robust Collator Devreye Giriyor
    custom_collator = DynamicRobustCollator(max_epochs=15)

    model = MMTD_Advanced()

    args = TrainingArguments(
        output_dir='/kaggle/working/results',
        num_train_epochs=15,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_acc",
        greater_is_better=True,
        learning_rate=1e-5,
        lr_scheduler_type="cosine",
        fp16=True,
        remove_unused_columns=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=custom_collator,
        compute_metrics=metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            PlotMetricsCallback(output_dir='/kaggle/working/results'),
            NoiseSchedulerCallback() # ⚠️ Gürültü Zamanlayıcı Eklendi
        ]
    )

    print("🚀 Training started with Dynamic Noise Scheduling...")
    trainer.train()
    trainer.save_model('/kaggle/working/MMTD_Final_Model')
    print("✅ Robust Training Completed!")

In [ ]:
%%writefile /kaggle/working/MMTD/utils.py
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, auc, roc_curve, f1_score, precision_score, recall_score
import numpy as np
import matplotlib.pyplot as plt
import torch
import seaborn as sb
import pandas as pd
import yaml
import os

def save_config(config: dict, save_path: str):
    with open(save_path, 'w') as f:
        f.write(yaml.dump(config, allow_unicode=True))

def metrics(eval_pred):
    """
    Hugging Face Trainer için özelleştirilmiş metrik fonksiyonu.
    F1-Score anahtarı 'f1_score' olarak belirlenmiştir.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    precision = precision_score(labels, predictions, average='macro', zero_division=0)
    recall = recall_score(labels, predictions, average='macro', zero_division=0)
    
    return {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    }

class EvalMetrics:
    """Eğitim sonrası detaylı ROC ve Heatmap analizi için sınıf."""
    def __init__(self, save_path=None, save_name=None, heatmap=True):
        self.save_path = save_path
        self.save_name = save_name
        self.heatmap = heatmap

    def __call__(self, eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        scores = torch.softmax(torch.from_numpy(logits).float(), dim=1).numpy()

        classes = ['ham', 'spam']
        # ROC Çizimi
        plt.figure(figsize=(8, 6))
        for i in range(len(classes)):
            fpr, tpr, _ = roc_curve(labels, scores[:, i], pos_label=i)
            plt.plot(fpr, tpr, label=f'ROC {classes[i]} (AUC = {auc(fpr, tpr):.4f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.title(f'ROC Curve - {self.save_name}')
        plt.legend(); plt.show()

        # Confusion Matrix
        if self.heatmap:
            plt.figure(figsize=(6, 5))
            sb.heatmap(confusion_matrix(labels, predictions), annot=True, fmt='d', cmap='Blues')
            plt.title('Confusion Matrix'); plt.show()

        return {"accuracy": accuracy_score(labels, predictions)}

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from models import MMTD_Advanced
from transformers import (
    Trainer, TrainingArguments, EarlyStoppingCallback, 
    TrainerCallback, BertTokenizerFast, ViTImageProcessor
)
from utils import metrics
from sklearn.model_selection import train_test_split

# --- GÜRÜLTÜ MODÜLLERİ (Hata Düzeltilmiş Sürüm) ---
def add_gaussian_noise(image, std=25):
    img_array = np.array(image).astype(np.float32) 
    noise = np.random.normal(0, std, img_array.shape)
    noisy_img = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_img)

def add_text_noise(text, noise_level=0.1):
    if not text or noise_level <= 0: return text
    chars = list(str(text))
    for _ in range(int(len(chars) * noise_level)):
        chars[random.randint(0, len(chars) - 1)] = random.choice("abcdefgh0123456789!@#$")
    return "".join(chars)

# --- DİNAMİK VERİ İŞLEYİCİ (Collator) ---
class DynamicRobustCollator:
    def __init__(self, max_epochs=16):
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.feature_extractor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
        self.current_epoch = 0
        self.max_epochs = max_epochs
        self.max_noise_prob = 0.7 

    def __call__(self, features):
        noise_prob = (self.current_epoch / self.max_epochs) * self.max_noise_prob
        texts, images, labels = [], [], []
        for f in features:
            txt, img, lbl = f["text"], f["image"], f["label"]
            if random.random() < noise_prob:
                txt = add_text_noise(txt)
                img = add_gaussian_noise(img)
            texts.append(txt); images.append(img); labels.append(lbl)

        inputs = self.tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
        pixel_values = self.feature_extractor(images, return_tensors="pt")["pixel_values"]
        return {
            "input_ids": inputs["input_ids"], "attention_mask": inputs["attention_mask"],
            "pixel_values": pixel_values, "labels": torch.tensor(labels, dtype=torch.long)
        }

# --- CALLBACKS ---
class NoiseSchedulerCallback(TrainerCallback):
    def __init__(self, collator): self.collator = collator
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.collator.current_epoch = int(state.epoch)
        print(f"\n🔔 Epoch {int(state.epoch)} - Gürültü İhtimali: %{(int(state.epoch)/16)*70:.1f}")

class PlotMetricsCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.train_loss, self.eval_loss, self.eval_f1 = [], [], []
        os.makedirs(self.output_dir, exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        if "loss" in logs: self.train_loss.append(logs["loss"])
        if "eval_loss" in logs:
            self.eval_loss.append(logs["eval_loss"])
            self.eval_f1.append(logs.get("eval_f1_score", 0))
            self.plot_graphs(state.epoch)

    def plot_graphs(self, epoch):
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, label='Train Loss', color='blue')
        plt.title(f'Loss Curve (Epoch {epoch:.0f})'); plt.legend(); plt.grid(True)
        plt.subplot(1, 2, 2)
        plt.plot(self.eval_f1, label='Eval F1-Score', color='orange')
        plt.title(f'F1-Score (Epoch {epoch:.0f})'); plt.legend(); plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, 'training_metrics_live.png'))
        plt.show(block=False); plt.pause(0.1); plt.close()

# --- EĞİTİM ---
if __name__ == "__main__":
    df = pd.read_csv('DATA/email_data/EDP.csv').rename(columns={"texts":"text","pics":"image","labels":"label"}).fillna("")
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

    from Email_dataset import EDPDataset
    train_set, test_set = EDPDataset('DATA/email_data/pics', train_df), EDPDataset('DATA/email_data/pics', test_df)
    
    collator = DynamicRobustCollator(max_epochs=16)
    model = MMTD_Advanced()

    t_args = TrainingArguments(
        output_dir='/kaggle/working/results',
        num_train_epochs=16,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        evaluation_strategy="epoch", # Kesin çözüm
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,
        learning_rate=1e-5,
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model, args=t_args, train_dataset=train_set, eval_dataset=test_set,
        data_collator=collator, compute_metrics=metrics,
        callbacks=[EarlyStoppingCallback(3), PlotMetricsCallback('/kaggle/working/results'), NoiseSchedulerCallback(collator)]
    )

    print("🚀 Robust Training v2.0 Başlıyor...")
    trainer.train()
    trainer.save_model('/kaggle/working/RobustMMTD_Final')

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from models import MMTD_Advanced
from transformers import (
    Trainer, TrainingArguments, EarlyStoppingCallback, 
    TrainerCallback, BertTokenizerFast, ViTImageProcessor
)
from utils import metrics
from sklearn.model_selection import train_test_split

# =====================================================================
# ⚡ HIZLI GÜRÜLTÜ MODÜLÜ (Albumentations ile)
# PIL tabanlı yavaş işlemler yerine C++ destekli Albumentations kullanıyoruz.
# =====================================================================
def get_robust_transforms(std_value=25):
    return A.Compose([
        A.GaussNoise(var_limit=(std_value**2, std_value**2), p=1.0),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)), # ViT standartları
        ToTensorV2()
    ])

def add_text_noise_fast(text, noise_level=0.1):
    if not text or noise_level <= 0: return text
    chars = list(str(text))
    num_noise = int(len(chars) * noise_level)
    for _ in range(num_noise):
        idx = random.randint(0, len(chars) - 1)
        chars[idx] = random.choice("abcdefgh0123456789!@#$")
    return "".join(chars)

# =====================================================================
# 🛡️ OPTİMİZE EDİLMİŞ COLLATOR (Multi-Processing Dostu)
# =====================================================================
class DynamicRobustCollator:
    def __init__(self, max_epochs=16):
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.current_epoch = 0
        self.max_epochs = max_epochs
        # Normal transform (Noise yok)
        self.clean_transform = A.Compose([
            A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
            ToTensorV2()
        ])
        # Gürültülü transform
        self.noisy_transform = get_robust_transforms()

    def __call__(self, features):
        noise_prob = (self.current_epoch / self.max_epochs) * 0.7
        texts, pixel_values_list, labels = [], [], []
        
        for f in features:
            txt, img, lbl = f["text"], f["image"], f["label"]
            img_np = np.array(img.convert("RGB")) # Bir kez çevir
            
            if random.random() < noise_prob:
                txt = add_text_noise_fast(txt)
                transformed = self.noisy_transform(image=img_np)["image"]
            else:
                transformed = self.clean_transform(image=img_np)["image"]
            
            texts.append(txt)
            pixel_values_list.append(transformed)
            labels.append(lbl)

        inputs = self.tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "pixel_values": torch.stack(pixel_values_list),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

# =====================================================================
# 📉 CALLBACKS (Zamanlayıcı ve Hafif Plotting)
# =====================================================================
class NoiseSchedulerCallback(TrainerCallback):
    def __init__(self, collator): self.collator = collator
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.collator.current_epoch = int(state.epoch)

class PlotMetricsCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.train_loss, self.eval_f1 = [], []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        if "loss" in logs: self.train_loss.append(logs["loss"])
        if "eval_f1_score" in logs:
            self.eval_f1.append(logs["eval_f1_score"])
            # Sadece 2 epochta bir plot çizerek overhead'i azaltıyoruz
            if int(state.epoch) % 2 == 0: self.plot_graphs(state.epoch)

    def plot_graphs(self, epoch):
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1); plt.plot(self.train_loss, color='blue'); plt.title("Loss")
        plt.subplot(1, 2, 2); plt.plot(self.eval_f1, color='orange'); plt.title("F1")
        plt.savefig(os.path.join(self.output_dir, 'training_metrics_live.png'))
        plt.close()

# =====================================================================
# 🚀 ÜRETİM (PRODUCTION) PIPELINE
# =====================================================================
if __name__ == "__main__":
    df = pd.read_csv('DATA/email_data/EDP.csv').rename(columns={"texts":"text","pics":"image","labels":"label"}).fillna("")
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

    from Email_dataset import EDPDataset
    train_set = EDPDataset('DATA/email_data/pics', train_df)
    test_set = EDPDataset('DATA/email_data/pics', test_df)
    
    collator = DynamicRobustCollator(max_epochs=16)
    model = MMTD_Advanced()

    # ❄️ OPSİYONEL: ViT ENCODER FREEZE (Hızı %50 artırmak istersen)
    # for param in model.image_encoder.parameters():
    #     param.requires_grad = False

    t_args = TrainingArguments(
        output_dir='/kaggle/working/results',
        num_train_epochs=16,
        
        # ⚡ BATCH OPTİMİZASYONU: Daha az step, daha çok paralel işlem
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        
        # 🏎️ DATALOADER HIZLANDIRICILAR
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=True,
        
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,
        learning_rate=2e-5, # Batch büyüdüğü için LR bir tık arttı
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model, args=t_args, train_dataset=train_set, eval_dataset=test_set,
        data_collator=collator, compute_metrics=metrics,
        callbacks=[EarlyStoppingCallback(3), PlotMetricsCallback('/kaggle/working/results'), NoiseSchedulerCallback(collator)]
    )

    print("🚀 Fırtına Modu (Lightning-Fast Training) Başlıyor...")
    trainer.train()
    trainer.save_model('/kaggle/working/RobustMMTD_Optimized')

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, ViTModel
from transformers.modeling_outputs import SequenceClassifierOutput


class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2, dropout_prob=0.2):
        super().__init__()

        # --- Encoders ---
        self.text_encoder = BertModel.from_pretrained("bert-base-multilingual-cased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        hidden = 768

        # --- Projections ---
        self.text_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU()
        )

        self.image_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU()
        )

        # --- Lightweight uncertainty (sequence-level, faster) ---
        self.text_uncertainty = nn.Linear(hidden, 1)
        self.image_uncertainty = nn.Linear(hidden, 1)

        # --- Modality scaling ---
        self.modality_scale = nn.Parameter(torch.tensor([1.0, 1.0]))

        # --- Fusion gate ---
        self.fusion_gate = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden, hidden),
            nn.Sigmoid()
        )

        # --- Fusion network ---
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden, hidden)
        )

        self.classifier = nn.Linear(hidden, num_labels)
        self.dropout = nn.Dropout(dropout_prob)

    # -------------------------
    # Helpers
    # -------------------------
    def uncertainty_weight(self, feat, head):
        # sequence-level uncertainty (FAST)
        u = torch.sigmoid(head(feat.mean(dim=1))).unsqueeze(1)
        return feat * (1.0 - u)

    def modality_dropout(self, x, p):
        if not self.training or p <= 0:
            return x
        mask = (torch.rand(x.size(0), 1, 1, device=x.device) > p).float()
        return x * mask

    # -------------------------
    # Forward
    # -------------------------
    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        labels=None,
        modality_dropout_prob=0.15,
        **kwargs
    ):

        # --- Encoding ---
        text_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        image_out = self.image_encoder(pixel_values=pixel_values)

        # --- CLS token pooling (FAST & STANDARD) ---
        text_feat = self.text_proj(text_out.last_hidden_state)
        image_feat = self.image_proj(image_out.last_hidden_state[:, 0])  # CLS only

        # Expand image to match text pipeline shape
        image_feat = image_feat.unsqueeze(1).expand(-1, text_feat.size(1), -1)

        # --- Uncertainty filtering ---
        text_feat = self.uncertainty_weight(text_feat, self.text_uncertainty)
        image_feat = self.uncertainty_weight(image_feat, self.image_uncertainty)

        # --- Modality dropout (batch-wise) ---
        text_feat = self.modality_dropout(text_feat, modality_dropout_prob)
        image_feat = self.modality_dropout(image_feat, modality_dropout_prob)

        # --- Pooling ---
        text_repr = (text_feat * attention_mask.unsqueeze(-1)).sum(1) / (attention_mask.sum(1, keepdim=True) + 1e-6)
        image_repr = image_feat.mean(dim=1)

        # --- Modality weighting ---
        weights = F.softmax(self.modality_scale, dim=0)
        text_repr = text_repr * weights[0]
        image_repr = image_repr * weights[1]

        # --- Fusion ---
        fused_input = torch.cat([text_repr, image_repr], dim=-1)

        gate = self.fusion_gate(fused_input)
        fused = self.fusion(fused_input) * gate + fused_input * (1 - gate)

        logits = self.classifier(self.dropout(fused))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels.view(-1).long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, ViTModel
from transformers.modeling_outputs import SequenceClassifierOutput


class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2, dropout_prob=0.2, train_last_vit_layers=4):
        super().__init__()

        # =========================================================
        # 🌐 Encoders
        # =========================================================
        self.text_encoder = BertModel.from_pretrained(
            "bert-base-multilingual-cased"
        )

        self.image_encoder = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        hidden = 768

        # =========================================================
        # 🧊 PARTIAL VIT FREEZING (IMPORTANT)
        # =========================================================
        self._freeze_vit(train_last_vit_layers)

        # =========================================================
        # 🔗 Projections
        # =========================================================
        self.text_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU()
        )

        self.image_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU()
        )

        # =========================================================
        # ⚖️ Uncertainty heads (lightweight)
        # =========================================================
        self.text_uncertainty = nn.Linear(hidden, 1)
        self.image_uncertainty = nn.Linear(hidden, 1)

        # =========================================================
        # 📊 Modality weighting
        # =========================================================
        self.modality_scale = nn.Parameter(torch.tensor([1.0, 1.0]))

        # =========================================================
        # 🔥 Fusion
        # =========================================================
        self.fusion_gate = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden, hidden),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden, hidden)
        )

        self.classifier = nn.Linear(hidden, num_labels)
        self.dropout = nn.Dropout(dropout_prob)

    # =========================================================
    # 🧊 VI T PARTIAL FREEZE
    # =========================================================
    def _freeze_vit(self, train_last_n):
        """
        ViT base = 12 layer
        first (12 - n) frozen, last n trainable
        """
        for name, param in self.image_encoder.named_parameters():
            if "encoder.layer" in name:
                layer_id = int(name.split(".")[2])

                total_layers = 12
                if layer_id < total_layers - train_last_n:
                    param.requires_grad = False
                else:
                    param.requires_grad = True

    # =========================================================
    # 🧠 Helpers
    # =========================================================
    def uncertainty_weight(self, feat, head):
        u = torch.sigmoid(head(feat.mean(dim=1))).unsqueeze(1)
        return feat * (1.0 - u)

    def modality_dropout(self, x, p):
        if not self.training or p <= 0:
            return x
        mask = (torch.rand(x.size(0), 1, 1, device=x.device) > p).float()
        return x * mask

    # =========================================================
    # 🚀 Forward
    # =========================================================
    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        labels=None,
        modality_dropout_prob=0.15,
        **kwargs
    ):

        # -------------------------
        # Text encoding
        # -------------------------
        text_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        text_feat = self.text_proj(text_out.last_hidden_state)

        # -------------------------
        # Image encoding (ViT CLS)
        # -------------------------
        image_out = self.image_encoder(pixel_values=pixel_values)

        image_feat = self.image_proj(
            image_out.last_hidden_state[:, 0]  # CLS token
        )

        image_feat = image_feat.unsqueeze(1).expand(-1, text_feat.size(1), -1)

        # -------------------------
        # Uncertainty filtering
        # -------------------------
        text_feat = self.uncertainty_weight(text_feat, self.text_uncertainty)
        image_feat = self.uncertainty_weight(image_feat, self.image_uncertainty)

        # -------------------------
        # Modality dropout
        # -------------------------
        text_feat = self.modality_dropout(text_feat, modality_dropout_prob)
        image_feat = self.modality_dropout(image_feat, modality_dropout_prob)

        # -------------------------
        # Pooling
        # -------------------------
        text_repr = (
            text_feat * attention_mask.unsqueeze(-1)
        ).sum(1) / (attention_mask.sum(1, keepdim=True) + 1e-6)

        image_repr = image_feat.mean(dim=1)

        # -------------------------
        # Modality weighting
        # -------------------------
        weights = F.softmax(self.modality_scale, dim=0)

        text_repr = text_repr * weights[0]
        image_repr = image_repr * weights[1]

        # -------------------------
        # Fusion
        # -------------------------
        fused_input = torch.cat([text_repr, image_repr], dim=-1)

        gate = self.fusion_gate(fused_input)
        fused = self.fusion(fused_input) * gate + fused_input * (1 - gate)

        logits = self.classifier(self.dropout(fused))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels.view(-1).long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from models import MMTD_Advanced
from transformers import (
    Trainer, TrainingArguments, EarlyStoppingCallback,
    TrainerCallback, BertTokenizerFast
)
from sklearn.model_selection import train_test_split
from utils import metrics


# =========================================================
# 📊 METRICS PLOT
# =========================================================
class PlotMetricsCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.train_loss = []
        self.eval_f1 = []
        os.makedirs(self.output_dir, exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        if "loss" in logs:
            self.train_loss.append(logs["loss"])

        # esnek key desteği
        for k in ["eval_f1_score", "eval_f1", "eval_f1"]:
            if k in logs:
                self.eval_f1.append(logs[k])

        if len(self.eval_f1) > 0 and state.epoch is not None:
            if int(state.epoch) % 1 == 0:
                self.plot(state.epoch)

    def plot(self, epoch):
        plt.figure(figsize=(10, 4))

        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, color="blue")
        plt.title("Loss")

        plt.subplot(1, 2, 2)
        plt.plot(self.eval_f1, color="orange")
        plt.title("F1 Score")

        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "training_metrics.png"))
        plt.close()


# =========================================================
# 🌪️ COLLATOR (NOISE + SAFE PIPELINE)
# =========================================================
class DynamicRobustCollator:
    def __init__(self, max_epochs=16):
        self.tokenizer = BertTokenizerFast.from_pretrained(
            "bert-base-multilingual-cased"
        )

        self.clean_tf = A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

        self.noisy_tf = A.Compose([
            A.Resize(224, 224),
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

        self.current_epoch = 0
        self.max_epochs = max_epochs

    def __call__(self, features):

        noise_prob = (self.current_epoch / self.max_epochs) * 0.7

        texts, images, labels = [], [], []

        for f in features:
            txt, img, lbl = f["text"], f["image"], f["label"]

            img = np.array(img.convert("RGB"))

            if random.random() < noise_prob:
                img = self.noisy_tf(image=img)["image"]
            else:
                img = self.clean_tf(image=img)["image"]

            texts.append(txt)
            images.append(img)
            labels.append(lbl)

        tokens = self.tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"],
            "attention_mask": tokens["attention_mask"],
            "pixel_values": torch.stack(images).float(),
            "labels": torch.tensor(labels, dtype=torch.long)
        }


# =========================================================
# 🔄 NOISE SCHEDULER
# =========================================================
class NoiseScheduler(TrainerCallback):
    def __init__(self, collator):
        self.collator = collator

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.collator.current_epoch = int(state.epoch)


# =========================================================
# 🚀 MAIN
# =========================================================
if __name__ == "__main__":

    df = pd.read_csv("DATA/email_data/EDP.csv")
    df = df.rename(columns={"texts": "text",
                            "pics": "image",
                            "labels": "label"}).fillna("")

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    from Email_dataset import EDPDataset

    train_set = EDPDataset("DATA/email_data/pics", train_df)
    test_set = EDPDataset("DATA/email_data/pics", test_df)

    collator = DynamicRobustCollator(max_epochs=16)
    model = MMTD_Advanced()

    # =====================================================
    # 🧊 VI T PARTIAL FREEZE (MODEL İÇİN)
    # =====================================================
    # model içinde zaten tanımlı

    args = TrainingArguments(
        output_dir="/kaggle/working/results",

        num_train_epochs=16,

        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,

        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=True,

        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,

        load_best_model_at_end=True,

        # esnek metric
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,

        learning_rate=2e-5,
        fp16=True,

        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_set,
        eval_dataset=test_set,
        data_collator=collator,
        compute_metrics=metrics,
        callbacks=[
            EarlyStoppingCallback(3),
            NoiseScheduler(collator),
            PlotMetricsCallback("/kaggle/working/results")
        ]
    )

    print("🚀 Eğitim Başladı (FINAL OPTIMIZED VERSION)")
    trainer.train()

    trainer.save_model("/kaggle/working/MMTD_FINAL")

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from models import MMTD_Advanced
from transformers import (
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    BertTokenizerFast
)
from sklearn.model_selection import train_test_split
from utils import metrics


# =========================================================
# 📊 METRICS GRAPH
# =========================================================
class PlotMetricsCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.train_loss = []
        self.eval_f1 = []
        os.makedirs(self.output_dir, exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        if "loss" in logs:
            self.train_loss.append(logs["loss"])

        # esnek F1 key
        for k in ["eval_f1_score", "eval_f1"]:
            if k in logs:
                self.eval_f1.append(logs[k])

        if state.epoch is not None and int(state.epoch) % 1 == 0:
            self.plot()

    def plot(self):
        plt.figure(figsize=(10, 4))

        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, color="blue")
        plt.title("Loss")

        plt.subplot(1, 2, 2)
        plt.plot(self.eval_f1, color="orange")
        plt.title("F1 Score")

        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "training_metrics.png"))
        plt.close()


# =========================================================
# 🌪️ COLLATOR (SAFE + FAST)
# =========================================================
class DynamicRobustCollator:
    def __init__(self, max_epochs=16):
        self.tokenizer = BertTokenizerFast.from_pretrained(
            "bert-base-multilingual-cased"
        )

        # CLEAN
        self.clean_tf = A.Compose([
            A.Resize(224, 224),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
            ToTensorV2()
        ])

        # NOISY
        self.noisy_tf = A.Compose([
            A.Resize(224, 224),
            A.GaussNoise(var_limit=(10, 50), p=1.0),  # FIXED
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
            ToTensorV2()
        ])

        self.current_epoch = 0
        self.max_epochs = max_epochs

    def __call__(self, features):

        noise_prob = (self.current_epoch / self.max_epochs) * 0.7

        texts, images, labels = [], [], []

        for f in features:
            txt, img, lbl = f["text"], f["image"], f["label"]

            img = np.array(img.convert("RGB"))

            if random.random() < noise_prob:
                img = self.noisy_tf(image=img)["image"]
            else:
                img = self.clean_tf(image=img)["image"]

            texts.append(txt)
            images.append(img)
            labels.append(lbl)

        tokens = self.tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"],
            "attention_mask": tokens["attention_mask"],
            "pixel_values": torch.stack(images).float(),
            "labels": torch.tensor(labels, dtype=torch.long)
        }


# =========================================================
# 🔄 NOISE SCHEDULER
# =========================================================
class NoiseScheduler(TrainerCallback):
    def __init__(self, collator):
        self.collator = collator

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.collator.current_epoch = int(state.epoch)


# =========================================================
# 🚀 MAIN
# =========================================================
if __name__ == "__main__":

    df = pd.read_csv("DATA/email_data/EDP.csv")
    df = df.rename(columns={
        "texts": "text",
        "pics": "image",
        "labels": "label"
    }).fillna("")

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    from Email_dataset import EDPDataset

    train_set = EDPDataset("DATA/email_data/pics", train_df)
    test_set = EDPDataset("DATA/email_data/pics", test_df)

    collator = DynamicRobustCollator(max_epochs=16)
    model = MMTD_Advanced()

    # =====================================================
    # 🧊 MODEL (partial freeze model içinde)
    # =====================================================

    args = TrainingArguments(
        output_dir="/kaggle/working/results",

        num_train_epochs=16,

        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,

        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=True,

        # FIXED (eski transformers uyumu)
        eval_strategy="epoch",
        save_strategy="epoch",

        logging_steps=100,

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,

        learning_rate=2e-5,
        fp16=True,

        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_set,
        eval_dataset=test_set,
        data_collator=collator,
        compute_metrics=metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            NoiseScheduler(collator),
            PlotMetricsCallback("/kaggle/working/results")
        ]
    )

    print("🚀 Training Started (FINAL FIXED VERSION)")
    trainer.train()

    trainer.save_model("/kaggle/working/MMTD_FINAL")
    

In [ ]:
# Email_dataset.py içindeki __getitem__ kısmını şu şekilde değiştir:
def __getitem__(self, idx):
    try:
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['label_name'], row['image'])
        image = Image.open(img_path).convert('RGB')
        
        # SÖZLÜK OLARAK DÖNDÜRÜYORUZ (Kritik nokta)
        return {
            "text": str(row['text']),
            "image": image,
            "label": int(row['label'])
        }
    except Exception as e:
        # Eğer resim okunamazsa (Çince karakter vs), listeden başka bir rastgele örnek döndür
        return self.__getitem__(random.randint(0, len(self.df)-1))

In [ ]:
%%writefile /kaggle/working/MMTD/Email_dataset.py
import os
import random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class EDPDataset(Dataset):
    def __init__(self, root_dir, df):
        self.root_dir = root_dir
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]
            # Sınıf klasörünü (ham/spam) belirle
            label_folder = "spam" if int(row['label']) == 1 else "ham"
            img_path = os.path.join(self.root_dir, label_folder, row['image'])
            
            # Dosya kontrolü (Çince karakterli dosyalar için önlem)
            if not os.path.exists(img_path):
                raise FileNotFoundError(f"Resim bulunamadı: {img_path}")

            image = Image.open(img_path).convert('RGB')
            
            # 🔥 KRİTİK DÜZELTME: Veriyi SÖZLÜK olarak döndür
            return {
                "text": str(row['text']),
                "image": image,
                "label": int(row['label'])
            }
        except Exception as e:
            # Eğer resim bozuksa veya isimden dolayı okunmuyorsa rastgele başka bir örnek getir
            return self.__getitem__(random.randint(0, len(self.df) - 1))

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from models import MMTD_Advanced
from transformers import (
    Trainer, TrainingArguments, EarlyStoppingCallback,
    TrainerCallback, BertTokenizerFast
)
from sklearn.model_selection import train_test_split
from utils import metrics

# =========================================================
# 📊 METRICS GRAPH (Geliştirilmiş Takip)
# =========================================================
class PlotMetricsCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.train_loss = []
        self.eval_f1 = []
        os.makedirs(self.output_dir, exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        if "loss" in logs: self.train_loss.append(logs["loss"])
        
        # Farklı versiyonlar için esnek key kontrolü
        f1_key = next((k for k in ["eval_f1_score", "eval_f1"] if k in logs), None)
        if f1_key:
            self.eval_f1.append(logs[f1_key])
            if int(state.epoch) % 1 == 0:
                self.plot_graphs(state.epoch)

    def plot_graphs(self, epoch):
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, color="blue", label="Train Loss")
        plt.title(f"Loss (Epoch {epoch:.0f})"); plt.grid(True)
        
        plt.subplot(1, 2, 2)
        plt.plot(self.eval_f1, color="orange", label="Eval F1")
        plt.title(f"F1-Score (Epoch {epoch:.0f})"); plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "training_metrics.png"))
        plt.close()

# =========================================================
# 🛡️ COLLATOR (Sözlük Uyumlu + Robust)
# =========================================================
class DynamicRobustCollator:
    def __init__(self, max_epochs=16):
        self.tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")
        self.clean_tf = A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        self.noisy_tf = A.Compose([
            A.Resize(224, 224),
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        self.current_epoch = 0
        self.max_epochs = max_epochs

    def __call__(self, features):
        noise_prob = (self.current_epoch / self.max_epochs) * 0.7
        texts, images, labels = [], [], []

        for f in features:
            # f artık bir sözlük olduğu için f["text"] hata vermez
            txt, img, lbl = f["text"], f["image"], f["label"]
            img_np = np.array(img)

            if random.random() < noise_prob:
                processed_img = self.noisy_tf(image=img_np)["image"]
            else:
                processed_img = self.clean_tf(image=img_np)["image"]

            texts.append(txt)
            images.append(processed_img)
            labels.append(lbl)

        tokens = self.tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
        
        return {
            "input_ids": tokens["input_ids"],
            "attention_mask": tokens["attention_mask"],
            "pixel_values": torch.stack(images).float(),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

class NoiseScheduler(TrainerCallback):
    def __init__(self, collator): self.collator = collator
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.collator.current_epoch = int(state.epoch)

# =========================================================
# 🚀 MAIN EXECUTION
# =========================================================
if __name__ == "__main__":
    df = pd.read_csv("DATA/email_data/EDP.csv").rename(columns={"texts":"text","pics":"image","labels":"label"}).fillna("")
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

    from Email_dataset import EDPDataset
    train_set = EDPDataset("DATA/email_data/pics", train_df)
    test_set = EDPDataset("DATA/email_data/pics", test_df)

    collator = DynamicRobustCollator(max_epochs=16)
    model = MMTD_Advanced() # models.py'da ViTModel.from_pretrained("google/vit-base-patch16-224") olmalı!

    args = TrainingArguments(
        output_dir="/kaggle/working/results",
        num_train_epochs=16,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        eval_strategy="epoch", # En güncel parametre ismi
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,
        learning_rate=2e-5,
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_set, eval_dataset=test_set,
        data_collator=collator, compute_metrics=metrics,
        callbacks=[EarlyStoppingCallback(3), NoiseScheduler(collator), PlotMetricsCallback("/kaggle/working/results")]
    )

    print("🚀 Eğitim Başladı (Kaggle Production v3.0)")
    trainer.train()
    trainer.save_model("/kaggle/working/MMTD_FINAL")

In [ ]:

import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from models import MMTD_Advanced
from transformers import (
    Trainer, TrainingArguments, EarlyStoppingCallback,
    TrainerCallback, BertTokenizerFast
)
from sklearn.model_selection import train_test_split
from utils import metrics

# =========================================================
# 📊 METRICS GRAPH (Geliştirilmiş Takip)
# =========================================================
class PlotMetricsCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.train_loss = []
        self.eval_f1 = []
        os.makedirs(self.output_dir, exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        if "loss" in logs: self.train_loss.append(logs["loss"])
        
        # Farklı versiyonlar için esnek key kontrolü
        f1_key = next((k for k in ["eval_f1_score", "eval_f1"] if k in logs), None)
        if f1_key:
            self.eval_f1.append(logs[f1_key])
            if int(state.epoch) % 1 == 0:
                self.plot_graphs(state.epoch)

    def plot_graphs(self, epoch):
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, color="blue", label="Train Loss")
        plt.title(f"Loss (Epoch {epoch:.0f})"); plt.grid(True)
        
        plt.subplot(1, 2, 2)
        plt.plot(self.eval_f1, color="orange", label="Eval F1")
        plt.title(f"F1-Score (Epoch {epoch:.0f})"); plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "training_metrics.png"))
        plt.close()

# =========================================================
# 🛡️ COLLATOR (Sözlük Uyumlu + Robust)
# =========================================================
class DynamicRobustCollator:
    def __init__(self, max_epochs=16):
        self.tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")
        self.clean_tf = A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        self.noisy_tf = A.Compose([
            A.Resize(224, 224),
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        self.current_epoch = 0
        self.max_epochs = max_epochs

    def __call__(self, features):
        noise_prob = (self.current_epoch / self.max_epochs) * 0.7
        texts, images, labels = [], [], []

        for f in features:
            # f artık bir sözlük olduğu için f["text"] hata vermez
            txt, img, lbl = f["text"], f["image"], f["label"]
            img_np = np.array(img)

            if random.random() < noise_prob:
                processed_img = self.noisy_tf(image=img_np)["image"]
            else:
                processed_img = self.clean_tf(image=img_np)["image"]

            texts.append(txt)
            images.append(processed_img)
            labels.append(lbl)

        tokens = self.tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
        
        return {
            "input_ids": tokens["input_ids"],
            "attention_mask": tokens["attention_mask"],
            "pixel_values": torch.stack(images).float(),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

class NoiseScheduler(TrainerCallback):
    def __init__(self, collator): self.collator = collator
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.collator.current_epoch = int(state.epoch)

# =========================================================
# 🚀 MAIN EXECUTION
# =========================================================
if __name__ == "__main__":
    df = pd.read_csv("DATA/email_data/EDP.csv").rename(columns={"texts":"text","pics":"image","labels":"label"}).fillna("")
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

    from Email_dataset import EDPDataset
    train_set = EDPDataset("DATA/email_data/pics", train_df)
    test_set = EDPDataset("DATA/email_data/pics", test_df)

    collator = DynamicRobustCollator(max_epochs=16)
    model = MMTD_Advanced() # models.py'da ViTModel.from_pretrained("google/vit-base-patch16-224") olmalı!

    args = TrainingArguments(
        output_dir="/kaggle/working/results",
        num_train_epochs=16,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        eval_strategy="epoch", # En güncel parametre ismi
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,
        learning_rate=2e-5,
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_set, eval_dataset=test_set,
        data_collator=collator, compute_metrics=metrics,
        callbacks=[EarlyStoppingCallback(3), NoiseScheduler(collator), PlotMetricsCallback("/kaggle/working/results")]
    )

    print("🚀 Eğitim Başladı (Kaggle Production v3.0)")
    trainer.train()
    trainer.save_model("/kaggle/working/MMTD_FINAL")

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from models import MMTD_Advanced
from transformers import (
    Trainer, TrainingArguments, EarlyStoppingCallback,
    TrainerCallback, BertTokenizerFast
)
from sklearn.model_selection import train_test_split
from utils import metrics
from torch.utils.data import Dataset

# =========================================================
# 📦 DATASET (Recursion-Free & Path-Fixed)
# =========================================================
class EDPDataset(Dataset):
    def __init__(self, root_dir, df):
        self.root_dir = root_dir
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        attempts = 0
        max_attempts = 10  # Maksimum 10 farklı resim denesin
        
        while attempts < max_attempts:
            try:
                # Rastgele bir index seç (ilk denemede orijinal indexi kullan)
                current_idx = idx if attempts == 0 else random.randint(0, len(self.df) - 1)
                row = self.df.iloc[current_idx]
                
                # 🔥 YOL DÜZELTME: Eğer CSV'de zaten 'ham/xxx.jpg' yazıyorsa 
                # sadece root_dir ile birleştiriyoruz.
                img_path = os.path.join(self.root_dir, str(row['image']))
                
                # Eğer yol hala yanlışsa (örn: 'ham/ham/' oluşuyorsa) alternatif kontrol:
                if not os.path.exists(img_path):
                    # CSV'de klasör ismi yoksa ekleyelim
                    label_folder = "spam" if int(row['label']) == 1 else "ham"
                    img_path = os.path.join(self.root_dir, label_folder, os.path.basename(str(row['image'])))

                if not os.path.exists(img_path):
                    raise FileNotFoundError(f"Resim yok: {img_path}")

                image = Image.open(img_path).convert('RGB')
                
                return {
                    "text": str(row['text']),
                    "image": image,
                    "label": int(row['label'])
                }
            except Exception:
                attempts += 1
        
        # Eğer hiçbir resim bulunamazsa boş bir görsel döndür (Eğitimi çökertmez)
        return {
            "text": "empty",
            "image": Image.new('RGB', (224, 224), (0, 0, 0)),
            "label": 0
        }

# =========================================================
# 🌪️ COLLATOR (Optimized)
# =========================================================
class DynamicRobustCollator:
    def __init__(self, max_epochs=16):
        self.tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")
        # Standart ViT normalizasyonu
        self.common_tf = A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        self.current_epoch = 0
        self.max_epochs = max_epochs

    def __call__(self, features):
        noise_prob = (self.current_epoch / self.max_epochs) * 0.7
        texts, images, labels = [], [], []

        for f in features:
            txt, img, lbl = f["text"], f["image"], f["label"]
            img_np = np.array(img)
            
            # Gürültüyü anlık (stochastically) ekle
            if random.random() < noise_prob:
                # Gaussian Noise ekle
                noise = np.random.normal(0, 25, img_np.shape)
                img_np = np.clip(img_np + noise, 0, 255).astype(np.uint8)

            processed_img = self.common_tf(image=img_np)["image"]
            texts.append(txt)
            images.append(processed_img)
            labels.append(lbl)

        tokens = self.tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
        return {
            "input_ids": tokens["input_ids"],
            "attention_mask": tokens["attention_mask"],
            "pixel_values": torch.stack(images).float(),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

# =========================================================
# 🔄 CALLBACKS
# =========================================================
class TrainingMonitor(TrainerCallback):
    def __init__(self, collator, output_dir):
        self.collator = collator
        self.output_dir = output_dir
        self.train_loss, self.eval_f1 = [], []
        os.makedirs(self.output_dir, exist_ok=True)

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.collator.current_epoch = int(state.epoch)
        print(f"\n🔔 Epoch {int(state.epoch)} Aktif - Gürültü Seviyesi Artıyor...")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        if "loss" in logs: self.train_loss.append(logs["loss"])
        f1_key = next((k for k in ["eval_f1_score", "eval_f1", "eval_acc"] if k in logs), None)
        if f1_key:
            self.eval_f1.append(logs[f1_key])
            self.plot()

    def plot(self):
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1); plt.plot(self.train_loss); plt.title("Loss")
        plt.subplot(1, 2, 2); plt.plot(self.eval_f1, color='orange'); plt.title("Performance")
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "live_metrics.png"))
        plt.close()

# =========================================================
# 🚀 START
# =========================================================
if __name__ == "__main__":
    df = pd.read_csv("DATA/email_data/EDP.csv").rename(columns={"texts":"text","pics":"image","labels":"label"}).fillna("")
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

    train_set = EDPDataset("DATA/email_data/pics", train_df)
    test_set = EDPDataset("DATA/email_data/pics", test_df)

    collator = DynamicRobustCollator(max_epochs=16)
    model = MMTD_Advanced()

    args = TrainingArguments(
        output_dir="/kaggle/working/results",
        num_train_epochs=16,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        dataloader_num_workers=2, # Hata ayıklama için worker sayısını 2'ye çektik
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,
        learning_rate=2e-5,
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model, args=args, train_dataset=train_set, eval_dataset=test_set,
        data_collator=collator, compute_metrics=metrics,
        callbacks=[EarlyStoppingCallback(3), TrainingMonitor(collator, "/kaggle/working/results")]
    )

    print("🚀 Fırtına Öncesi Sessizlik... Eğitim Başlıyor!")
    trainer.train()
    trainer.save_model("/kaggle/working/MMTD_FINAL")

In [ ]:
%%writefile /kaggle/working/MMTD/Email_dataset.py

import os
from PIL import Image
from torch.utils.data import Dataset


class EDPDataset(Dataset):

    def __init__(self, root_dir, dataframe):
        self.root_dir = root_dir
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        text = str(row["text"])
        label = int(row["label"])

        img_path = os.path.join(
            self.root_dir,
            str(row["image"])
        )

        try:
            image = Image.open(img_path).convert("RGB")

        except Exception as e:
            print(f"⚠️ Resim okunamadı: {img_path} | {e}")

            # dummy image fallback
            image = Image.new("RGB", (224, 224), color=(0, 0, 0))

        return {
            "text": text,
            "image": image,
            "label": label
        }

In [ ]:
%%writefile /kaggle/working/MMTD/Email_dataset.py

import os
from PIL import Image
from torch.utils.data import Dataset


class EDPDataset(Dataset):

    def __init__(self, root_dir, dataframe):
        self.root_dir = root_dir
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        text = str(row["text"])
        label = int(row["label"])

        img_path = os.path.join(
            self.root_dir,
            str(row["image"])
        )

        try:
            image = Image.open(img_path).convert("RGB")

        except Exception as e:
            print(f"⚠️ Resim okunamadı: {img_path} | {e}")

            # dummy image fallback
            image = Image.new("RGB", (224, 224), color=(0, 0, 0))

        return {
            "text": text,
            "image": image,
            "label": label
        }

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from models import MMTD_Advanced
from transformers import (
    Trainer, TrainingArguments, EarlyStoppingCallback,
    TrainerCallback, BertTokenizerFast
)
from sklearn.model_selection import train_test_split
from utils import metrics
from Email_dataset import EDPDataset # Senin verdiğin yeni sözlük yapılı dataset

# =========================================================
# 📊 METRICS & NOISE MONITOR
# =========================================================
class TrainingMonitor(TrainerCallback):
    def __init__(self, collator, output_dir):
        self.collator = collator
        self.output_dir = output_dir
        self.train_loss = []
        self.eval_f1 = []
        os.makedirs(self.output_dir, exist_ok=True)

    def on_epoch_begin(self, args, state, control, **kwargs):
        # Her epoch başında gürültü ihtimalini güncelle
        self.collator.current_epoch = int(state.epoch)
        prob = (self.collator.current_epoch / self.collator.max_epochs) * 0.7
        print(f"\n🔔 Epoch {int(state.epoch)} Başlıyor - Mevcut Gürültü İhtimali: %{prob*100:.1f}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        if "loss" in logs: self.train_loss.append(logs["loss"])
        
        # Farklı versiyonlar için esnek key kontrolü (eval_f1_score veya eval_f1)
        f1_key = next((k for k in ["eval_f1_score", "eval_f1"] if k in logs), None)
        if f1_key:
            self.eval_f1.append(logs[f1_key])
            self.plot_metrics(state.epoch)

    def plot_metrics(self, epoch):
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, color="blue", label="Train Loss")
        plt.title(f"Loss (Epoch {epoch:.0f})"); plt.grid(True)
        
        plt.subplot(1, 2, 2)
        plt.plot(self.eval_f1, color="orange", label="F1 Score")
        plt.title(f"F1-Score (Epoch {epoch:.0f})"); plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "training_metrics.png"))
        plt.close()

# =========================================================
# 🌪️ DİNAMİK GÜRÜLTÜCÜ (Collator)
# =========================================================
class DynamicRobustCollator:
    def __init__(self, max_epochs=16):
        self.tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")
        # ViT Standart Normalizasyonu
        self.clean_tf = A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        # Gürültülü Transform
        self.noisy_tf = A.Compose([
            A.Resize(224, 224),
            A.GaussNoise(p=1.0), 
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        self.current_epoch = 0
        self.max_epochs = max_epochs

    def __call__(self, features):
        noise_prob = (self.current_epoch / self.max_epochs) * 0.7
        texts, images, labels = [], [], []

        for f in features:
            # Email_dataset.py artık dict döndürdüğü için bu kısım hatasız çalışacak
            txt, img, lbl = f["text"], f["image"], f["label"]
            img_np = np.array(img)

            if random.random() < noise_prob:
                processed_img = self.noisy_tf(image=img_np)["image"]
            else:
                processed_img = self.clean_tf(image=img_np)["image"]

            texts.append(txt)
            images.append(processed_img)
            labels.append(lbl)

        tokens = self.tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
        
        return {
            "input_ids": tokens["input_ids"],
            "attention_mask": tokens["attention_mask"],
            "pixel_values": torch.stack(images).float(),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

# =========================================================
# 🚀 ANA EĞİTİM DÖNGÜSÜ
# =========================================================
if __name__ == "__main__":
    # Veriyi yükle ve temizle
    df = pd.read_csv("DATA/email_data/EDP.csv").rename(columns={"texts":"text","pics":"image","labels":"label"}).fillna("")
    
    # Stratified split (Dengeli sınıflar için)
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

    # Datasetleri oluştur
    train_set = EDPDataset("DATA/email_data/pics", train_df)
    test_set = EDPDataset("DATA/email_data/pics", test_df)

    collator = DynamicRobustCollator(max_epochs=16)
    model = MMTD_Advanced()

    args = TrainingArguments(
        output_dir="/kaggle/working/results",
        num_train_epochs=16,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        
        # Hız Optimizasyonu
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        
        # Metrik Ayarları
        eval_strategy="epoch", # Güncel versiyon ismi
        save_strategy="epoch",
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,
        
        # Donanım ve Optimizasyon
        learning_rate=2e-5,
        fp16=True,
        
        # 🔥 KRİTİK: 'text' ve 'image' anahtarlarının silinmesini engeller
        remove_unused_columns=False, 
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_set,
        eval_dataset=test_set,
        data_collator=collator,
        compute_metrics=metrics,
        callbacks=[
            EarlyStoppingCallback(3),
            TrainingMonitor(collator, "/kaggle/working/results")
        ]
    )

    print("🚀 MMTD Robust Training Pipeline Başlatılıyor...")
    trainer.train()
    
    # Final Modelini Kaydet
    trainer.save_model("/kaggle/working/MMTD_FINAL_MODEL")
    print("✅ Eğitim tamamlandı ve model kaydedildi!")

In [ ]:
# =========================================================
# 🚀 FINAL STABLE VERSION - ROBUST MMTD TRAINING PIPELINE
# =========================================================

import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A

from albumentations.pytorch import ToTensorV2
from PIL import Image

import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    BertModel,
    ViTModel,
    BertTokenizerFast,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback
)

from transformers.modeling_outputs import SequenceClassifierOutput

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# =========================================================
# ⚡ SPEED OPTIMIZATION
# =========================================================

torch.backends.cudnn.benchmark = True

# =========================================================
# 📊 METRICS
# =========================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)

    f1 = f1_score(labels, predictions, average="macro")

    return {
        "eval_accuracy": acc,
        "eval_f1_score": f1
    }

# =========================================================
# 🧠 MODEL
# =========================================================

class MMTD_Advanced(nn.Module):

    def __init__(self, num_labels=2):

        super().__init__()

        # -------------------------------------------------
        # TEXT ENCODER
        # -------------------------------------------------

        self.text_encoder = BertModel.from_pretrained(
            "bert-base-multilingual-cased"
        )

        # -------------------------------------------------
        # IMAGE ENCODER
        # -------------------------------------------------

        self.image_encoder = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        # =================================================
        # 🔒 PARTIAL FREEZING
        # =================================================

        # TEXT
        for param in self.text_encoder.parameters():
            param.requires_grad = False

        for i in [10, 11]:
            for param in self.text_encoder.encoder.layer[i].parameters():
                param.requires_grad = True

        for param in self.text_encoder.pooler.parameters():
            param.requires_grad = True

        # IMAGE
        for param in self.image_encoder.parameters():
            param.requires_grad = False

        for i in [10, 11]:
            for param in self.image_encoder.encoder.layer[i].parameters():
                param.requires_grad = True

        hidden = 768

        # =================================================
        # PROJECTION
        # =================================================

        self.text_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU()
        )

        self.image_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU()
        )

        # =================================================
        # UNCERTAINTY HEADS
        # =================================================

        self.text_uncertainty = nn.Linear(hidden, 1)
        self.image_uncertainty = nn.Linear(hidden, 1)

        # =================================================
        # FUSION
        # =================================================

        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        # =================================================
        # CLASSIFIER
        # =================================================

        self.classifier = nn.Linear(hidden, num_labels)

        self.dropout = nn.Dropout(0.2)

    # =====================================================
    # FORWARD
    # =====================================================

    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        labels=None,
        **kwargs
    ):

        # -------------------------------------------------
        # TEXT FEATURES
        # -------------------------------------------------

        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        text_feat = text_outputs.last_hidden_state[:, 0]

        # -------------------------------------------------
        # IMAGE FEATURES
        # -------------------------------------------------

        image_outputs = self.image_encoder(
            pixel_values=pixel_values
        )

        image_feat = image_outputs.last_hidden_state[:, 0]

        # -------------------------------------------------
        # PROJECTION
        # -------------------------------------------------

        text_feat = self.text_proj(text_feat)

        image_feat = self.image_proj(image_feat)

        # -------------------------------------------------
        # UNCERTAINTY WEIGHTING
        # -------------------------------------------------

        text_weight = 1.0 - torch.sigmoid(
            self.text_uncertainty(text_feat)
        )

        image_weight = 1.0 - torch.sigmoid(
            self.image_uncertainty(image_feat)
        )

        text_feat = text_feat * text_weight

        image_feat = image_feat * image_weight

        # -------------------------------------------------
        # FUSION
        # -------------------------------------------------

        fused = torch.cat(
            [text_feat, image_feat],
            dim=-1
        )

        fused = self.fusion(fused)

        logits = self.classifier(
            self.dropout(fused)
        )

        # -------------------------------------------------
        # LOSS
        # -------------------------------------------------

        loss = None

        if labels is not None:

            loss = F.cross_entropy(
                logits,
                labels.view(-1).long()
            )

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits
        )

# =========================================================
# 📦 DATASET
# =========================================================

class EDPDataset(torch.utils.data.Dataset):

    def __init__(self, root_dir, dataframe):

        self.root_dir = root_dir

        self.df = dataframe.reset_index(drop=True)

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        text = str(row["text"])

        label = int(row["label"])

        image_name = str(row["image"])

        # -------------------------------------------------
        # PATH FIX
        # -------------------------------------------------

        path1 = os.path.join(
            self.root_dir,
            image_name
        )

        label_folder = "spam" if label == 1 else "ham"

        path2 = os.path.join(
            self.root_dir,
            label_folder,
            os.path.basename(image_name)
        )

        img_path = path1 if os.path.exists(path1) else path2

        # -------------------------------------------------
        # IMAGE LOAD
        # -------------------------------------------------

        try:

            image = Image.open(img_path).convert("RGB")

        except Exception:

            # fallback image
            image = Image.new(
                "RGB",
                (224, 224),
                (0, 0, 0)
            )

        return {
            "text": text,
            "image": image,
            "label": label
        }

# =========================================================
# 🛡️ DYNAMIC COLLATOR
# =========================================================

class DynamicCollator:

    def __init__(self, max_epochs=16):

        self.tokenizer = BertTokenizerFast.from_pretrained(
            "bert-base-multilingual-cased"
        )

        self.current_epoch = 0

        self.max_epochs = max_epochs

        self.transform = A.Compose([

            A.Resize(224, 224),

            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),

            ToTensorV2()

        ])

    def add_text_noise(self, text):

        chars = list(text)

        if len(chars) < 5:
            return text

        n = max(1, int(len(chars) * 0.05))

        for _ in range(n):

            idx = random.randint(0, len(chars) - 1)

            chars[idx] = random.choice(
                "abcdefghijklmnopqrstuvwxyz0123456789"
            )

        return "".join(chars)

    def __call__(self, features):

        noise_prob = (
            self.current_epoch / self.max_epochs
        ) * 0.7

        texts = []

        images = []

        labels = []

        for f in features:

            text = f["text"]

            image = f["image"]

            label = f["label"]

            img_np = np.array(image)

            # -------------------------------------------------
            # NOISE SCHEDULING
            # -------------------------------------------------

            if random.random() < noise_prob:

                # IMAGE NOISE
                noise = np.random.normal(
                    0,
                    20,
                    img_np.shape
                )

                img_np = np.clip(
                    img_np + noise,
                    0,
                    255
                ).astype(np.uint8)

                # TEXT NOISE
                text = self.add_text_noise(text)

            img_tensor = self.transform(
                image=img_np
            )["image"]

            texts.append(text)

            images.append(img_tensor)

            labels.append(label)

        tokens = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {

            "input_ids": tokens["input_ids"],

            "attention_mask": tokens["attention_mask"],

            "pixel_values": torch.stack(images),

            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )
        }

# =========================================================
# 📉 CALLBACKS
# =========================================================

class EpochCallback(TrainerCallback):

    def __init__(self, collator):

        self.collator = collator

    def on_epoch_begin(
        self,
        args,
        state,
        control,
        **kwargs
    ):

        self.collator.current_epoch = int(state.epoch)

        prob = (
            self.collator.current_epoch / 16
        ) * 70

        print(
            f"\n🔔 Epoch {int(state.epoch)} "
            f"- Gürültü Oranı: %{prob:.1f}"
        )

# =========================================================
# 📊 PLOT CALLBACK
# =========================================================

class PlotCallback(TrainerCallback):

    def __init__(self):

        self.train_loss = []

        self.eval_f1 = []

    def on_log(
        self,
        args,
        state,
        control,
        logs=None,
        **kwargs
    ):

        if logs is None:
            return

        if "loss" in logs:

            self.train_loss.append(logs["loss"])

        if "eval_f1_score" in logs:

            self.eval_f1.append(
                logs["eval_f1_score"]
            )

            plt.figure(figsize=(10, 4))

            plt.subplot(1, 2, 1)

            plt.plot(self.train_loss)

            plt.title("Training Loss")

            plt.grid(True)

            plt.subplot(1, 2, 2)

            plt.plot(self.eval_f1)

            plt.title("Validation F1 Score")

            plt.grid(True)

            plt.tight_layout()

            plt.savefig(
                "training_metrics.png"
            )

            plt.close()

# =========================================================
# 🚀 TRAINING
# =========================================================

if __name__ == "__main__":

    # -----------------------------------------------------
    # LOAD CSV
    # -----------------------------------------------------

    df = pd.read_csv(
        "DATA/email_data/EDP.csv"
    )

    df = df.rename(columns={

        "texts": "text",

        "pics": "image",

        "labels": "label"

    }).fillna("")

    # -----------------------------------------------------
    # SPLIT
    # -----------------------------------------------------

    train_df, test_df = train_test_split(

        df,

        test_size=0.2,

        stratify=df["label"],

        random_state=42
    )

    # -----------------------------------------------------
    # DATASETS
    # -----------------------------------------------------

    train_set = EDPDataset(
        "DATA/email_data/pics",
        train_df
    )

    test_set = EDPDataset(
        "DATA/email_data/pics",
        test_df
    )

    # -----------------------------------------------------
    # COLLATOR
    # -----------------------------------------------------

    collator = DynamicCollator(
        max_epochs=16
    )

    # -----------------------------------------------------
    # MODEL
    # -----------------------------------------------------

    model = MMTD_Advanced()

    # -----------------------------------------------------
    # TRAINING ARGS
    # -----------------------------------------------------

    args = TrainingArguments(

        output_dir="./results",

        num_train_epochs=16,

        per_device_train_batch_size=8,

        gradient_accumulation_steps=2,

        dataloader_num_workers=2,

        dataloader_pin_memory=True,

        eval_strategy="epoch",

        save_strategy="epoch",

        save_total_limit=2,

        logging_steps=50,

        load_best_model_at_end=True,

        metric_for_best_model="eval_f1_score",

        greater_is_better=True,

        fp16=torch.cuda.is_available(),

        remove_unused_columns=False,

        report_to="none"
    )

    # -----------------------------------------------------
    # TRAINER
    # -----------------------------------------------------

    trainer = Trainer(

        model=model,

        args=args,

        train_dataset=train_set,

        eval_dataset=test_set,

        data_collator=collator,

        compute_metrics=compute_metrics,

        callbacks=[

            EarlyStoppingCallback(
                early_stopping_patience=3
            ),

            EpochCallback(collator),

            PlotCallback()

        ]
    )

    # -----------------------------------------------------
    # START
    # -----------------------------------------------------

    print(
        "🚀 FINAL ROBUST MMTD TRAINING STARTED"
    )

    trainer.train()

    # -----------------------------------------------------
    # SAVE FINAL MODEL
    # -----------------------------------------------------

    trainer.save_model(
        "./MMTD_FINAL"
    )

    print(
        "✅ TRAINING COMPLETED SUCCESSFULLY"
    )

In [ ]:
import shutil
# MMTD_FINAL klasörünü zipleyip tek bir dosya haline getirir
shutil.make_archive('/kaggle/working/MMTD_En_Iyi_Model', 'zip', './MMTD_FINAL')

In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BertModel, ViTModel
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from safetensors.torch import load_file

# =====================================================================
# 1. METRİKLER VE GÜRÜLTÜ FONKSİYONLARI
# =====================================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"eval_acc": acc, "eval_f1_score": f1}

def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        if len(text) > 0:
            index = random.randint(0, len(text) - 1)
            noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

# =====================================================================
# 2. 🧠 MODEL MİMARİSİ (Ağırlıklarla Birebir Uyumlu Şablon)
# =====================================================================
class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2):
        super().__init__()
        self.text_encoder = BertModel.from_pretrained("bert-base-multilingual-cased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        hidden = 768
        # Projeksiyonlar (Eğittiğimiz haliyle tam uyumlu Sequential yapısı)
        self.text_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.image_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU())
        
        # Uncertainty Heads
        self.text_uncertainty = nn.Linear(hidden, 1)
        self.image_uncertainty = nn.Linear(hidden, 1)
        
        # Fusion
        self.fusion = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(hidden, num_labels)
        self.dropout = nn.Dropout(0.2)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, **kwargs):
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = text_outputs.last_hidden_state[:, 0]
        
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        image_feat = image_outputs.last_hidden_state[:, 0]

        text_feat = self.text_proj(text_feat)
        image_feat = self.image_proj(image_feat)

        text_weight = 1.0 - torch.sigmoid(self.text_uncertainty(text_feat))
        image_weight = 1.0 - torch.sigmoid(self.image_uncertainty(image_feat))

        text_feat = text_feat * text_weight
        image_feat = image_feat * image_weight

        fused = torch.cat([text_feat, image_feat], dim=-1)
        fused = self.fusion(fused)
        logits = self.classifier(self.dropout(fused))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels.view(-1).long())
        return SequenceClassifierOutput(loss=loss, logits=logits)

# =====================================================================
# 3. KUSURSUZ DATASET & COLLATOR
# =====================================================================
class EDPDatasetDict(torch.utils.data.Dataset):
    def __init__(self, root_dir, dataframe):
        self.root_dir = root_dir
        self.df = dataframe.reset_index(drop=True)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path1 = os.path.join(self.root_dir, str(row["image"]))
        label_folder = "spam" if int(row["label"]) == 1 else "ham"
        path2 = os.path.join(self.root_dir, label_folder, os.path.basename(str(row["image"])))
        img_path = path1 if os.path.exists(path1) else path2

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), (0, 0, 0))
        return {"text": str(row["text"]), "image": image, "label": int(row["label"])}

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.transform = A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def __call__(self, features):
        texts, images, labels = [], [], []
        for f in features:
            t, p, lbl = f["text"], f["image"], f["label"]
            if self.text_noise_level > 0: t = add_text_noise(t, self.text_noise_level)
            if self.image_noise: p = add_gaussian_noise(p)
            
            img_tensor = self.transform(image=np.array(p))["image"]
            texts.append(t); images.append(img_tensor); labels.append(lbl)

        tokens = self.tokenizer(texts, return_tensors='pt', max_length=128, truncation=True, padding='max_length')
        return {"input_ids": tokens["input_ids"], "attention_mask": tokens["attention_mask"], "pixel_values": torch.stack(images), "labels": torch.tensor(labels, dtype=torch.long)}

# =====================================================================
# 🚀 4. TEST LABORATUVARI BAŞLIYOR
# =====================================================================
if __name__ == "__main__":
    print("🚨 Gürbüzlük (Robustness) Testi Başlıyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU).rename(columns={"texts": "text", "pics": "image", "labels": "label"}).fillna("")
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    test_dataset = EDPDatasetDict(RESIMLERIN_YOLU, test_df)

    print("🧠 Şampiyon model (MMTD_Advanced) uyandırılıyor...")
    model = MMTD_Advanced()
    
    # Kaggle'daki dataset yolundan ağırlıkları çekiyoruz
    model_path = '/kaggle/input/datasets/arafkubraa/mmtdrobustyenimimari/model.safetensors'
    model.load_state_dict(load_file(model_path))
    model.eval() 

    args = TrainingArguments(output_dir='./test_results', per_device_eval_batch_size=8, dataloader_num_workers=2, remove_unused_columns=False, report_to="none")

    scenarios = [
        {"name": "1. Temiz Veri (Baseline)", "text_noise": 0.0, "image_noise": False},
        {"name": "2. Sadece Resim Gürültüsü (Gaussian)", "text_noise": 0.0, "image_noise": True},
        {"name": "3. Sadece Metin Gürültüsü (%10)", "text_noise": 0.1, "image_noise": False},
        {"name": "4. Tam Kaos (Metin %10 + Resim)", "text_noise": 0.1, "image_noise": True},
    ]

    print("\n📊 --- HAKEMLER İÇİN KARŞILAŞTIRMALI TEST SONUÇLARI ---")
    for s in scenarios:
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        
        trainer = Trainer(
            model=model,
            args=args,
            eval_dataset=test_dataset,
            data_collator=collator,
            compute_metrics=compute_metrics
        )
        
        result = trainer.evaluate() 
        
        # Olası isim farklılıklarına karşı garanti çözüm:
        acc = result.get('eval_eval_acc', result.get('eval_acc', 0))
        f1 = result.get('eval_eval_f1_score', result.get('eval_f1_score', 0))
        
        print(f"✅ [{s['name']}] -> Başarı (Acc): %{acc*100:.2f} | F1 Skoru: %{f1*100:.2f} | Kayıp: {result['eval_loss']:.4f}\n")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 📊 TEZ İÇİN AKADEMİK GÜRBÜZLÜK GRAFİĞİ
# =========================================================

# Ekrandaki Başarı (F1) Verilerin
scenarios = ['Temiz Veri\n(Baseline)', 'Sadece Resim\nGürültüsü', 'Sadece Metin\nGürültüsü', 'Tam Kaos\n(Resim + Metin)']
f1_scores = [98.46, 98.22, 97.05, 96.68]

# Akademik, şık bir stil belirleyelim
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Çubuk grafiğini çiz (Renk paleti: viridis, sunumlar için çok şıktır)
ax = sns.barplot(x=scenarios, y=f1_scores, hue=scenarios, palette="viridis", legend=False)

# Farkları net göstermek için Y eksenini 90'dan başlatalım
plt.ylim(90, 100)

# Başlık ve Eksen İsimleri
plt.title('MMTD Advanced - Siber Saldırı (Robustness) Testi F1 Skorları', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('F1 Skoru (%)', fontsize=12, fontweight='bold')
plt.xlabel('Test Senaryoları', fontsize=12, fontweight='bold')

# Çubukların üzerine % değerlerini yazdıralım
for p in ax.patches:
    ax.annotate(f'%{p.get_height():.2f}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 10), 
                textcoords='offset points',
                fontsize=12, fontweight='bold', color='black')

# Düzeni toparla ve Yüksek Çözünürlüklü (300 dpi) kaydet
plt.tight_layout()
plt.savefig('/kaggle/working/robustness_results.png', dpi=300)

print("✅ Grafik başarıyla oluşturuldu ve 'robustness_results.png' olarak Output klasörüne kaydedildi!")
plt.show()

In [ ]:
# =========================================================
# 🚀 ABLATION STUDY - CLEAN BASELINE MMTD TRAINING
# =========================================================

import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A

from albumentations.pytorch import ToTensorV2
from PIL import Image

import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    BertModel,
    ViTModel,
    BertTokenizerFast,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback
)

from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# =========================================================
# ⚡ SPEED OPTIMIZATION
# =========================================================
torch.backends.cudnn.benchmark = True

# =========================================================
# 📊 METRICS
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"eval_accuracy": acc, "eval_f1_score": f1}

# =========================================================
# 🧠 MODEL (Mimari tamamen aynı kalıyor)
# =========================================================
class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2):
        super().__init__()
        self.text_encoder = BertModel.from_pretrained("bert-base-multilingual-cased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        # 🔒 PARTIAL FREEZING
        for param in self.text_encoder.parameters(): param.requires_grad = False
        for i in [10, 11]:
            for param in self.text_encoder.encoder.layer[i].parameters(): param.requires_grad = True
        for param in self.text_encoder.pooler.parameters(): param.requires_grad = True

        for param in self.image_encoder.parameters(): param.requires_grad = False
        for i in [10, 11]:
            for param in self.image_encoder.encoder.layer[i].parameters(): param.requires_grad = True

        hidden = 768

        self.text_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.image_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU())
        
        self.text_uncertainty = nn.Linear(hidden, 1)
        self.image_uncertainty = nn.Linear(hidden, 1)

        self.fusion = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(hidden, num_labels)
        self.dropout = nn.Dropout(0.2)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, **kwargs):
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = text_outputs.last_hidden_state[:, 0]

        image_outputs = self.image_encoder(pixel_values=pixel_values)
        image_feat = image_outputs.last_hidden_state[:, 0]

        text_feat = self.text_proj(text_feat)
        image_feat = self.image_proj(image_feat)

        text_weight = 1.0 - torch.sigmoid(self.text_uncertainty(text_feat))
        image_weight = 1.0 - torch.sigmoid(self.image_uncertainty(image_feat))

        text_feat = text_feat * text_weight
        image_feat = image_feat * image_weight

        fused = torch.cat([text_feat, image_feat], dim=-1)
        fused = self.fusion(fused)
        logits = self.classifier(self.dropout(fused))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels.view(-1).long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

# =========================================================
# 📦 DATASET
# =========================================================
class EDPDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, dataframe):
        self.root_dir = root_dir
        self.df = dataframe.reset_index(drop=True)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text, label, image_name = str(row["text"]), int(row["label"]), str(row["image"])

        path1 = os.path.join(self.root_dir, image_name)
        label_folder = "spam" if label == 1 else "ham"
        path2 = os.path.join(self.root_dir, label_folder, os.path.basename(image_name))
        img_path = path1 if os.path.exists(path1) else path2

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        return {"text": text, "image": image, "label": label}

# =========================================================
# 🛡️ CLEAN COLLATOR (Tüm gürültü işlemleri silindi!)
# =========================================================
class CleanCollator:
    def __init__(self):
        self.tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")
        self.transform = A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def __call__(self, features):
        texts, images, labels = [], [], []

        for f in features:
            # Gürültü eklemeden direkt alıyoruz
            texts.append(f["text"])
            
            # Görüntüyü sadece standart boyuta getirip tensor yapıyoruz
            img_np = np.array(f["image"])
            img_tensor = self.transform(image=img_np)["image"]
            images.append(img_tensor)
            
            labels.append(f["label"])

        tokens = self.tokenizer(
            texts, padding=True, truncation=True, max_length=128, return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"],
            "attention_mask": tokens["attention_mask"],
            "pixel_values": torch.stack(images),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

# =========================================================
# 📊 PLOT CALLBACK
# =========================================================
class PlotCallback(TrainerCallback):
    def __init__(self):
        self.train_loss = []
        self.eval_f1 = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        if "loss" in logs: self.train_loss.append(logs["loss"])
        if "eval_f1_score" in logs:
            self.eval_f1.append(logs["eval_f1_score"])
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.plot(self.train_loss)
            plt.title("Training Loss")
            plt.grid(True)
            plt.subplot(1, 2, 2)
            plt.plot(self.eval_f1)
            plt.title("Validation F1 Score")
            plt.grid(True)
            plt.tight_layout()
            plt.savefig("clean_training_metrics.png")
            plt.close()

# =========================================================
# 🚀 TRAINING
# =========================================================
if __name__ == "__main__":
    df = pd.read_csv("DATA/email_data/EDP.csv").rename(columns={"texts": "text", "pics": "image", "labels": "label"}).fillna("")
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

    train_set = EDPDataset("DATA/email_data/pics", train_df)
    test_set = EDPDataset("DATA/email_data/pics", test_df)

    collator = CleanCollator()
    model = MMTD_Advanced()

    args = TrainingArguments(
        output_dir="./clean_results",
        num_train_epochs=16, # Early stopping kesecek
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_set,
        eval_dataset=test_set,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            PlotCallback()
        ]
    )

    print("🚀 CLEAN BASELINE MMTD TRAINING STARTED (NO NOISE)")
    trainer.train()

    # KARIŞMAMASI İÇİN FARKLI İSİMLE KAYDEDİYORUZ
    trainer.save_model("./MMTD_CLEAN_BASELINE")
    print("✅ CLEAN TRAINING COMPLETED SUCCESSFULLY")

In [ ]:
import shutil
import os

# Eski ara kayıt (checkpoint) klasörlerini siliyoruz
klasorler = ['./results', './clean_results']

for k in klasorler:
    if os.path.exists(k):
        shutil.rmtree(k)
        print(f"✅ {k} klasörü silindi, diskte yer açıldı!")
    else:
        print(f"ℹ️ {k} zaten yok.")

In [ ]:
# =========================================================
# 🚀 ABLATION STUDY - CLEAN BASELINE MMTD TRAINING
# =========================================================

import os
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A

from albumentations.pytorch import ToTensorV2
from PIL import Image

import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    BertModel,
    ViTModel,
    BertTokenizerFast,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback
)

from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# =========================================================
# ⚡ SPEED OPTIMIZATION
# =========================================================
torch.backends.cudnn.benchmark = True

# =========================================================
# 📊 METRICS
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"eval_accuracy": acc, "eval_f1_score": f1}

# =========================================================
# 🧠 MODEL (Mimari tamamen aynı kalıyor)
# =========================================================
class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2):
        super().__init__()
        self.text_encoder = BertModel.from_pretrained("bert-base-multilingual-cased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        # 🔒 PARTIAL FREEZING
        for param in self.text_encoder.parameters(): param.requires_grad = False
        for i in [10, 11]:
            for param in self.text_encoder.encoder.layer[i].parameters(): param.requires_grad = True
        for param in self.text_encoder.pooler.parameters(): param.requires_grad = True

        for param in self.image_encoder.parameters(): param.requires_grad = False
        for i in [10, 11]:
            for param in self.image_encoder.encoder.layer[i].parameters(): param.requires_grad = True

        hidden = 768

        self.text_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.image_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU())
        
        self.text_uncertainty = nn.Linear(hidden, 1)
        self.image_uncertainty = nn.Linear(hidden, 1)

        self.fusion = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(hidden, num_labels)
        self.dropout = nn.Dropout(0.2)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, **kwargs):
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = text_outputs.last_hidden_state[:, 0]

        image_outputs = self.image_encoder(pixel_values=pixel_values)
        image_feat = image_outputs.last_hidden_state[:, 0]

        text_feat = self.text_proj(text_feat)
        image_feat = self.image_proj(image_feat)

        text_weight = 1.0 - torch.sigmoid(self.text_uncertainty(text_feat))
        image_weight = 1.0 - torch.sigmoid(self.image_uncertainty(image_feat))

        text_feat = text_feat * text_weight
        image_feat = image_feat * image_weight

        fused = torch.cat([text_feat, image_feat], dim=-1)
        fused = self.fusion(fused)
        logits = self.classifier(self.dropout(fused))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels.view(-1).long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

# =========================================================
# 📦 DATASET
# =========================================================
class EDPDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, dataframe):
        self.root_dir = root_dir
        self.df = dataframe.reset_index(drop=True)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text, label, image_name = str(row["text"]), int(row["label"]), str(row["image"])

        path1 = os.path.join(self.root_dir, image_name)
        label_folder = "spam" if label == 1 else "ham"
        path2 = os.path.join(self.root_dir, label_folder, os.path.basename(image_name))
        img_path = path1 if os.path.exists(path1) else path2

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        return {"text": text, "image": image, "label": label}

# =========================================================
# 🛡️ CLEAN COLLATOR (Tüm gürültü işlemleri silindi!)
# =========================================================
class CleanCollator:
    def __init__(self):
        self.tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")
        self.transform = A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def __call__(self, features):
        texts, images, labels = [], [], []

        for f in features:
            # Gürültü eklemeden direkt alıyoruz
            texts.append(f["text"])
            
            # Görüntüyü sadece standart boyuta getirip tensor yapıyoruz
            img_np = np.array(f["image"])
            img_tensor = self.transform(image=img_np)["image"]
            images.append(img_tensor)
            
            labels.append(f["label"])

        tokens = self.tokenizer(
            texts, padding=True, truncation=True, max_length=128, return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"],
            "attention_mask": tokens["attention_mask"],
            "pixel_values": torch.stack(images),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

# =========================================================
# 📊 PLOT CALLBACK
# =========================================================
class PlotCallback(TrainerCallback):
    def __init__(self):
        self.train_loss = []
        self.eval_f1 = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        if "loss" in logs: self.train_loss.append(logs["loss"])
        if "eval_f1_score" in logs:
            self.eval_f1.append(logs["eval_f1_score"])
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.plot(self.train_loss)
            plt.title("Training Loss")
            plt.grid(True)
            plt.subplot(1, 2, 2)
            plt.plot(self.eval_f1)
            plt.title("Validation F1 Score")
            plt.grid(True)
            plt.tight_layout()
            plt.savefig("clean_training_metrics.png")
            plt.close()

# =========================================================
# 🚀 TRAINING
# =========================================================
if __name__ == "__main__":
    df = pd.read_csv("DATA/email_data/EDP.csv").rename(columns={"texts": "text", "pics": "image", "labels": "label"}).fillna("")
    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

    train_set = EDPDataset("DATA/email_data/pics", train_df)
    test_set = EDPDataset("DATA/email_data/pics", test_df)

    collator = CleanCollator()
    model = MMTD_Advanced()

    args = TrainingArguments(
        output_dir="./clean_results",
        num_train_epochs=16, # Early stopping kesecek
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_score",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_set,
        eval_dataset=test_set,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            PlotCallback()
        ]
    )

    print("🚀 CLEAN BASELINE MMTD TRAINING STARTED (NO NOISE)")
    trainer.train()

    # KARIŞMAMASI İÇİN FARKLI İSİMLE KAYDEDİYORUZ
    trainer.save_model("./MMTD_CLEAN_BASELINE")
    print("✅ CLEAN TRAINING COMPLETED SUCCESSFULLY")

In [ ]:
import shutil

# Yeni eğittiğimiz "MMTD_CLEAN_BASELINE" klasörünü zipliyoruz
shutil.make_archive('/kaggle/working/MMTD_Clean_Baseline_Model', 'zip', './MMTD_CLEAN_BASELINE')

print("✅ Temiz model başarıyla ziplendi! Sağ panelden indirebilirsin.")

In [ ]:
from IPython.display import FileLink

# Hücrenin altına tıklanabilir bir indirme linki bırakır
FileLink('/kaggle/working/MMTD_Clean_Baseline_Model.zip')

In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import Trainer, TrainingArguments, BertTokenizerFast, BertModel, ViTModel
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from safetensors.torch import load_file

# =====================================================================
# 1. METRİKLER VE GÜRÜLTÜ FONKSİYONLARI
# =====================================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"eval_acc": acc, "eval_f1_score": f1}

def add_gaussian_noise(image, mean=0, std=25):
    img_array = np.array(image)
    noise = np.random.normal(mean, std, img_array.shape)
    noisy_image_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_image_array)

def add_text_noise(text, noise_level=0.1):
    if noise_level <= 0 or noise_level > 1: return text
    noisy_text = list(text)
    num_noise_chars = int(len(text) * noise_level)
    for _ in range(num_noise_chars):
        if len(text) > 0:
            index = random.randint(0, len(text) - 1)
            noisy_text[index] = random.choice("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890!@#$%^&*()_-+=[]{}|;:,.<>?")
    return ''.join(noisy_text)

# =====================================================================
# 2. 🧠 MODEL MİMARİSİ
# =====================================================================
class MMTD_Advanced(nn.Module):
    def __init__(self, num_labels=2):
        super().__init__()
        self.text_encoder = BertModel.from_pretrained("bert-base-multilingual-cased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        hidden = 768
        self.text_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.image_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU())
        
        self.text_uncertainty = nn.Linear(hidden, 1)
        self.image_uncertainty = nn.Linear(hidden, 1)
        
        self.fusion = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(hidden, num_labels)
        self.dropout = nn.Dropout(0.2)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, **kwargs):
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = text_outputs.last_hidden_state[:, 0]
        
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        image_feat = image_outputs.last_hidden_state[:, 0]

        text_feat = self.text_proj(text_feat)
        image_feat = self.image_proj(image_feat)

        text_weight = 1.0 - torch.sigmoid(self.text_uncertainty(text_feat))
        image_weight = 1.0 - torch.sigmoid(self.image_uncertainty(image_feat))

        text_feat = text_feat * text_weight
        image_feat = image_feat * image_weight

        fused = torch.cat([text_feat, image_feat], dim=-1)
        fused = self.fusion(fused)
        logits = self.classifier(self.dropout(fused))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels.view(-1).long())
        return SequenceClassifierOutput(loss=loss, logits=logits)

# =====================================================================
# 3. KUSURSUZ DATASET & ROBUST COLLATOR (Gürültü Enjekte Eden)
# =====================================================================
class EDPDatasetDict(torch.utils.data.Dataset):
    def __init__(self, root_dir, dataframe):
        self.root_dir = root_dir
        self.df = dataframe.reset_index(drop=True)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path1 = os.path.join(self.root_dir, str(row["image"]))
        label_folder = "spam" if int(row["label"]) == 1 else "ham"
        path2 = os.path.join(self.root_dir, label_folder, os.path.basename(str(row["image"])))
        img_path = path1 if os.path.exists(path1) else path2

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), (0, 0, 0))
        return {"text": str(row["text"]), "image": image, "label": int(row["label"])}

class EDPCollatorRobust:
    def __init__(self, text_noise_level=0.0, image_noise=False):
        self.text_noise_level = text_noise_level
        self.image_noise = image_noise
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.transform = A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def __call__(self, features):
        texts, images, labels = [], [], []
        for f in features:
            t, p, lbl = f["text"], f["image"], f["label"]
            if self.text_noise_level > 0: t = add_text_noise(t, self.text_noise_level)
            if self.image_noise: p = add_gaussian_noise(p)
            
            img_tensor = self.transform(image=np.array(p))["image"]
            texts.append(t); images.append(img_tensor); labels.append(lbl)

        tokens = self.tokenizer(texts, return_tensors='pt', max_length=128, truncation=True, padding='max_length')
        return {"input_ids": tokens["input_ids"], "attention_mask": tokens["attention_mask"], "pixel_values": torch.stack(images), "labels": torch.tensor(labels, dtype=torch.long)}

# =====================================================================
# 🚀 4. "BÜYÜK YÜZLEŞME" TEST LABORATUVARI
# =====================================================================
if __name__ == "__main__":
    print("🚨 Kırılgan (Baseline) Model Gürbüzlük Testine Giriyor...\n")

    DATA_CSV_YOLU = '/kaggle/working/DATA/email_data/EDP.csv' 
    RESIMLERIN_YOLU = '/kaggle/working/DATA/email_data/pics'
    
    # Klasör yolunu garantiye alalım (Önceki MMTD klasörü sorunu için)
    if not os.path.exists(DATA_CSV_YOLU):
        DATA_CSV_YOLU = '/kaggle/working/MMTD/DATA/email_data/EDP.csv' 
        RESIMLERIN_YOLU = '/kaggle/working/MMTD/DATA/email_data/pics'

    df = pd.read_csv(DATA_CSV_YOLU).rename(columns={"texts": "text", "pics": "image", "labels": "label"}).fillna("")
    _, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
    test_dataset = EDPDatasetDict(RESIMLERIN_YOLU, test_df)

    print("🧠 Temiz (Gürültüsüz Eğitilen) Model uyandırılıyor...")
    model = MMTD_Advanced()
    
    # 🎯 AZ ÖNCE EĞİTTİĞİMİZ TEMİZ MODELİN AĞIRLIKLARINI ÇEKİYORUZ
    # pytorch_model.bin veya model.safetensors olabilir, garanti olsun diye deniyoruz:
    baseline_path = '/kaggle/working/MMTD/MMTD_CLEAN_BASELINE/model.safetensors'
    if not os.path.exists(baseline_path):
        baseline_path = '/kaggle/working/MMTD/MMTD_CLEAN_BASELINE/pytorch_model.bin'
        model.load_state_dict(torch.load(baseline_path))
    else:
        model.load_state_dict(load_file(baseline_path))
        
    model.eval() 

    args = TrainingArguments(output_dir='./test_results', per_device_eval_batch_size=8, dataloader_num_workers=2, remove_unused_columns=False, report_to="none")

    scenarios = [
        {"name": "1. Temiz Veri (Baseline)", "text_noise": 0.0, "image_noise": False},
        {"name": "2. Sadece Resim Gürültüsü (Gaussian)", "text_noise": 0.0, "image_noise": True},
        {"name": "3. Sadece Metin Gürültüsü (%10)", "text_noise": 0.1, "image_noise": False},
        {"name": "4. Tam Kaos (Metin %10 + Resim)", "text_noise": 0.1, "image_noise": True},
    ]

    print("\n📊 --- JÜRİ İÇİN: TEMİZ MODELİN KAOS TESTİ SONUÇLARI ---")
    for s in scenarios:
        collator = EDPCollatorRobust(text_noise_level=s["text_noise"], image_noise=s["image_noise"])
        trainer = Trainer(model=model, args=args, eval_dataset=test_dataset, data_collator=collator, compute_metrics=compute_metrics)
        result = trainer.evaluate() 
        
        acc = result.get('eval_eval_acc', result.get('eval_acc', 0))
        f1 = result.get('eval_eval_f1_score', result.get('eval_f1_score', 0))
        
        print(f"✅ [{s['name']}] -> Başarı (Acc): %{acc*100:.2f} | F1 Skoru: %{f1*100:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# 📊 TEZ İÇİN KARŞILAŞTIRMALI ABLATION GRAFİĞİ
# =========================================================

# Test Senaryoları
labels = ['Temiz Veri\n(Baseline)', 'Sadece Resim\nGürültüsü', 'Sadece Metin\nGürültüsü', 'Tam Kaos\n(Resim + Metin)']

# Ekrandan aldığımız sonuçlar
clean_scores = [98.45, 97.31, 95.60, 92.61]  # Temiz Veriyle Eğitilen Model
robust_scores = [98.46, 98.22, 97.05, 96.68] # Gürültüyle Eğitilen (Şampiyon) Model

x = np.arange(len(labels))  # Etiketlerin x eksenindeki konumları
width = 0.35  # Çubuk genişliği

# Şık bir grafik figürü oluşturalım
fig, ax = plt.subplots(figsize=(12, 6))

# Çubukları çizdirme (Renkler: Kırılgan için Kırmızımsı, Gürbüz için Yeşile dönük Mavi)
rects1 = ax.bar(x - width/2, clean_scores, width, label='Temiz Eğitimli (Baseline)', color='#e74c3c')
rects2 = ax.bar(x + width/2, robust_scores, width, label='Gürbüz Eğitimli (Curriculum Learning)', color='#2980b9')

# Eksen yazıları ve başlık
ax.set_ylabel('F1 Skoru (%)', fontsize=12, fontweight='bold')
ax.set_title('MMTD Advanced - Eğitim Stratejilerinin Siber Saldırı (Gürültü) Altındaki Performans Farkı', fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11, fontweight='bold')
ax.legend(fontsize=11)

# Farkı dramatik ve net göstermek için Y eksenini 90'dan başlatalım
ax.set_ylim(90, 100)

# Çubukların üzerine değerleri yazdıralım
ax.bar_label(rects1, fmt='%%%.2f', padding=3, fontsize=11, fontweight='bold', color='#c0392b')
ax.bar_label(rects2, fmt='%%%.2f', padding=3, fontsize=11, fontweight='bold', color='#1f618d')

# Düzeni toparla ve Yüksek Çözünürlüklü kaydet
fig.tight_layout()
plt.savefig('/kaggle/working/ablation_karsilastirma.png', dpi=300)

print("✅ Muazzam karşılaştırma grafiği başarıyla çizildi ve kaydedildi!")
plt.show()